# 미션7 
- 이번 미션에서는 SSD 모델을 활용하여 개와 고양이의 얼굴(Face) 영역을 감지하는 Object Detection 작업을 수행해 봅니다.

### 데이터 소개
- The Oxford-IIIT Pet Dataset https://www.kaggle.com/datasets/devdgohil/the-oxfordiiit-pet-dataset 
    - annotations(xml): 각 이미지 파일에 대한 annotation
    - images: 이미지 파일(37종의 개와 고양이)

요구 사항
- 데이터 로드, 전처리, 모델 학습, 예측, 성능 평가 등의 모든 과정을 포함해야 합니다.
- Bounding Box에 대한 정보를 활용하여 SSD 모델을 활용한 Object Detection을 수행
- 마크다운에 코드의 각 단계에서 어떤 작업을 수행하는지, 어떤 의도를 가지고 접근했는지 명확히 표현
- 마크다운에 코드와 실행 결과를 설명하는 문구를 추가하여, 전체 코드의 흐름을 이해할 수 있도록 작성
- 보고서를 따로 작성하지 않으므로, 노트북 파일 내에 모든 설명
- 소스내 클래스 및 함수 설명을 주석으로 처리 
- 모델 성능 평가
    - 평가 지표(mAP, IOU 등)를 활용해 모델 성능을 분석하고 비교
    - 제공된 데이터셋의 테스트 파일을 사용하여 모델을 테스트
    - 모델별 성능 평가 결과를 노트북에 출력

### 전체 흐름

| 단계 | 내용 | 의도 |
|---|---|---|
| 1 | 환경 설정 | 재현성(seed), GPU/AMP, 실험 설정 한 곳(`CFG`)에 모으기 |
| 2 | Kaggle 접속 & 다운로드 | API 인증 → `kagglehub` 다운로드 → 폴더 자동 탐색 |
| 3 | EDA | XML 구조, 클래스(종·품종) 불균형, 박스 크기·위치·종횡비, 이미지 크기 |
| 4 | 데이터 분할 | XML 이 있는 이미지만 GT 를 가짐 → 품종 층화 train/val/test 분할 |
| 5 | 로드 & 전처리 | 메모리 캐시, 박스 좌표 변환, Dataset · 증강(v2 transforms, Mosaic) |
| 6 | 평가 지표 구현 | IoU, Precision/Recall, AP(101-point), mAP@0.5 / 0.75 / 0.5:0.95, 크기별 AP |
| 7 | 공통 유틸 | 모델 생성(헤드 교체), 학습 루프, 조기 종료, 과적합 진단 |
| 8 | 실험 1 | **Baseline SSD300-VGG16** (기본 증강) |
| 9 | 실험 2 | **고도화된 데이터 증강** (Photometric · ZoomOut · IoU-Crop · Mosaic) |
| 10 | 실험 3 | **소형 객체 탐지 개선** (앵커 커버리지 분석 → SSD512 + 작은 앵커) |
| 11 | 실험 4 | **클래스 불균형 해소** (고양이 오버샘플링) |
| 12 | 실험 5 | **아키텍처 · 손실함수 개선** (Focal + GIoU, SSDLite-MobileNetV3, YOLO 비교) |
| 13 | 검증셋 비교 | 지표·학습곡선·과적합 진단 |
| 14 | 최종 테스트 | 테스트셋으로 전 모델 Precision/Recall/AP/mAP 평가 |
| 15 | 후처리 개선 | NMS 임계값, Class-agnostic NMS, Soft-NMS, max-det |
| 16 | 추론 속도 · 경량화 | FP16, channels_last, 해상도, torch.compile, ONNX Runtime, 속도–정확도 비교 |
| 17 | 시각화 · 오류 분석 | 예측 박스, 오탐/미탐 사례, 공식 test 목록(정답 없음) 추론 |
| 18 | 결론 | 결과 요약 · 한계 |

---
## 1. 환경 설정

필요 라이브러리를 설치하고 불러온다. (torchvision 0.16+ 의 `transforms.v2` 가 박스를 함께 변환해 준다).

- `ultralytics` : YOLO 비교 실험용 (12장, 선택)
- `onnx`, `onnxruntime` : 추론 가속 실험용 (16장, 선택) — GPU 에서 ORT 를 쓰려면 `onnxruntime-gpu` 를 설치한다.

In [1]:
#%pip install -q kagglehub opencv-python-headless scikit-learn seaborn pandas matplotlib 
%pip install -q ultralytics onnx onnxruntime

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os, re, gc, json, time, copy, math, pickle, random, warnings, getpass, shutil
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict, Counter
from functools import partial
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import tv_tensors
from torchvision.transforms import v2 as T2
from torchvision.ops import box_iou, nms, batched_nms, generalized_box_iou_loss
from torchvision.models.detection import (ssd300_vgg16, SSD300_VGG16_Weights,
                                          ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights)
from torchvision.models.detection.ssd import SSDClassificationHead
from torchvision.models.detection.ssdlite import SSDLiteClassificationHead
from torchvision.models.detection.anchor_utils import DefaultBoxGenerator
from torchvision.models.detection.transform import GeneralizedRCNNTransform
from torchvision.models.detection import _utils as det_utils

from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
print("torch", torch.__version__, "| torchvision", torchvision.__version__)

torch 2.14.0+cpu | torchvision 0.29.0+cpu


### 1-1. 실험 설정 & 재현성

- 하이퍼파라미터를 `CFG` 한 곳에 모아 실험 조건을 명확히 하고, 실험마다 `seed_everything()` 으로 난수를 다시 고정해 **실험 간 비교를 공정**하게 한다.
- GPU 가 있으면 **AMP(Mixed Precision)** 로 학습 속도·메모리를 절약한다.
- Windows + Jupyter 에서는 DataLoader 멀티프로세싱이 노트북 내 클래스 pickling 문제를 일으키므로 `num_workers=0`.
- 학습률은 미션 6 과 같은 **Discriminative LR** : 새로 붙인 분류 헤드는 크게(1e-3), 사전학습된 백본·박스 헤드는 작게(1e-4).
- `LOCAL_DATA_DIR` : 데이터를 이미 받아 둔 경우 그 폴더를 지정하면 다운로드를 건너뛴다.

In [2]:
QUICK_MODE = False       # True: 데이터 일부 + 2 epoch (흐름 확인용)
PRETRAINED = True        # COCO 사전학습 가중치 사용
RUN_YOLO = True          # YOLO 비교 실험 (ultralytics 필요)
RUN_ONNX = True          # ONNX Runtime 가속 실험 (onnx, onnxruntime 필요)
LOCAL_DATA_DIR = None    # 예: r"C:\data\oxford-pet"  (None 이면 kagglehub 로 다운로드)

CFG = {
    "seed": 42,
    "cache_max_side": 512,      # 메모리 캐시 시 긴 변 길이 (SSD512 실험까지 고려)
    "batch_size": 32,
    "num_workers": 0 if os.name == "nt" else 2,
    "lr_head": 1e-3, "lr_backbone": 1e-4, "weight_decay": 1e-4,
    "warmup_epochs": 1,
    "patience": 6,              # EarlyStopping (val mAP@0.5:0.95 기준)
    "val_size": 0.15, "test_size": 0.15,
    "op_score": 0.5,            # Precision/Recall 계산용 운영 점수 임계값
    "max_images": None,         # 디버그용 이미지 수 제한
    "epochs": {"ssd": 24, "ssd512": 24, "ssdlite": 40, "yolo": 40},
    "yolo_models": ["yolo11n.pt", "yolo26n.pt"],   # NMS 기반(YOLO11) vs NMS-free(YOLO26)
    "yolo_imgsz": 320,
}
if QUICK_MODE:
    CFG["epochs"] = {k: 2 for k in CFG["epochs"]}
    CFG["max_images"] = 600
    CFG["patience"] = 99

OUT_DIR = Path("outputs_m7"); OUT_DIR.mkdir(exist_ok=True)
CLASSES = ["background", "cat", "dog"]          # SSD 는 0 번을 배경으로 사용
LABEL = {"cat": 1, "dog": 2}
CLASS_IDS = [1, 2]
NUM_CLASSES = len(CLASSES)
IOU_THRS = np.round(np.arange(0.5, 0.96, 0.05), 2)   # COCO mAP@[.5:.95]

def seed_everything(seed=CFG["seed"]):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything()
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print("device:", device, "| AMP:", USE_AMP)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

device: cpu | AMP: False


---
## 2. Kaggle 접속 및 데이터 다운로드

### 2-1. Kaggle API 인증 방법 체크
Kaggle → **Settings → API → Create New Token** 으로 받은 `kaggle.json` 을 `~/.kaggle/` 에 두거나, 환경변수 `KAGGLE_USERNAME` / `KAGGLE_KEY` 를 설정한다. 둘 다 없으면 아래 셀이 입력창으로 물어본다(Key 는 화면에 표시되지 않음).

In [3]:
KAGGLE_JSON = Path.home() / ".kaggle" / "kaggle.json"

if LOCAL_DATA_DIR:
    print("✔ 로컬 데이터 사용 → 인증 생략:", LOCAL_DATA_DIR)
elif KAGGLE_JSON.exists():
    print("✔ kaggle.json 인증 파일 사용:", KAGGLE_JSON)
elif os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
    print("✔ 환경변수(KAGGLE_USERNAME / KAGGLE_KEY) 인증 사용")
else:
    os.environ["KAGGLE_USERNAME"] = input("Kaggle Username: ").strip()
    os.environ["KAGGLE_KEY"] = getpass.getpass("Kaggle API Key: ").strip()
    print("✔ 입력한 계정 정보로 인증")

✔ kaggle.json 인증 파일 사용: C:\Users\dsbang\.kaggle\kaggle.json


### 2-2. 데이터셋 다운로드 & 파일 자동 탐색
`kagglehub.dataset_download()` 가 압축 해제된 경로를 돌려준다(약 800MB, 두 번째 실행부터 캐시 사용).

압축 안의 폴더 구조(`annotations/annotations/xmls`, `images/images` 처럼 중첩)가 버전마다 다를 수 있으므로, 경로를 하드코딩하지 않고 **확장자·파일명으로 자동 탐색**한다.

- `*.xml` : 얼굴 박스 annotation (PASCAL VOC 형식)
- `*.jpg` : 원본 이미지 (`trimaps` 폴더의 분할 마스크 png 는 제외)
- `list.txt`, `trainval.txt`, `test.txt` : 이미지별 품종 ID·종(species) 정보와 공식 분할 목록

In [4]:
print(LOCAL_DATA_DIR)

None


In [ ]:
if LOCAL_DATA_DIR:
    download_path = Path(LOCAL_DATA_DIR)
else:
    import kagglehub
    download_path = Path(kagglehub.dataset_download("devdgohil/the-oxfordiiit-pet-dataset"))
print("데이터 경로:", download_path)

def find_dataset_files(base):
    """다운로드 폴더 전체를 훑어 xml / 이미지 / 분할 목록 파일 위치를 찾는다.

    Returns:
        xmls  : {이미지 stem: xml 경로}
        imgs  : {이미지 stem: 이미지 경로}  (trimaps, __MACOSX, '._' 숨김파일 제외)
        lists : {'list': 경로, 'trainval': 경로, 'test': 경로} (없으면 키 없음)
    """
    base = Path(base)
    skip = lambda p: "__MACOSX" in p.parts or p.name.startswith("._") or "trimaps" in [s.lower() for s in p.parts]
    xmls, imgs, lists = {}, {}, {}
    for p in sorted(base.rglob("*"), key=lambda x: len(x.parts)):     # 얕은 경로 우선 (중복 폴더 대비)
        if not p.is_file() or skip(p):
            continue
        suf = p.suffix.lower()
        if suf == ".xml":
            xmls.setdefault(p.stem, p)
        elif suf in {".jpg", ".jpeg", ".png"}:
            imgs.setdefault(p.stem, p)
        elif p.name in {"list.txt", "trainval.txt", "test.txt"}:
            lists.setdefault(p.stem, p)
    return xmls, imgs, lists

XML_FILES, IMG_FILES, LIST_FILES = find_dataset_files(download_path)
print(f"XML {len(XML_FILES)}개 | 이미지 {len(IMG_FILES)}개 | 목록 파일 {list(LIST_FILES)}")
print("XML 예시 :", next(iter(XML_FILES.values())))
print("이미지 예시:", next(iter(IMG_FILES.values())))

**결과 해석** : XML 개수가 이미지 개수보다 훨씬 적다면, **얼굴 박스는 일부 이미지(공식 trainval 목록)에만 제공**된다는 뜻이다. 이 사실이 4장의 분할 전략을 결정한다.

---
## 3. 데이터 탐색 (EDA)

### 3-1. XML annotation 구조 확인
먼저 XML 한 개를 그대로 출력해 어떤 필드가 있는지 확인한다.

In [ ]:
sample_xml = next(iter(XML_FILES.values()))
print(sample_xml.read_text(encoding="utf-8", errors="ignore")[:1200])

**관찰** : PASCAL VOC 형식이다. `<size>` 에 이미지 크기, `<object>` 마다 `<name>`(cat / dog), `<pose>`, `<truncated>`(잘림), `<difficult>`(판별 어려움), `<bndbox>`(xmin, ymin, xmax, ymax 픽셀 좌표) 가 있다. 박스는 **몸 전체가 아니라 얼굴(head) 영역**이다.

### 3-2. 전체 XML 파싱 → DataFrame
- `ann_df` : 박스(객체) 1개당 1행 — 박스 크기·위치 분석용
- `img_df` : 이미지 1개당 1행 — 종·품종 분포, 분할용
- 품종(breed)은 파일명에서(`Abyssinian_100` → `Abyssinian`), 종(species)은 `list.txt`(1=cat, 2=dog)에서, 없으면 XML 의 `<name>` 에서 가져온다.

In [ ]:
def parse_voc_xml(path):
    """PASCAL VOC XML 1개를 파싱해 (이미지 정보 dict, 객체 dict 리스트)를 반환한다."""
    root = ET.parse(path).getroot()
    size = root.find("size")
    info = {"xml_w": int(float(size.findtext("width", 0))), "xml_h": int(float(size.findtext("height", 0)))}
    objs = []
    for o in root.findall("object"):
        bb = o.find("bndbox")
        objs.append({"name": o.findtext("name", "").strip().lower(),
                     "pose": o.findtext("pose", "Unspecified"),
                     "truncated": int(o.findtext("truncated", "0") or 0),
                     "difficult": int(o.findtext("difficult", "0") or 0),
                     **{k: float(bb.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax")}})
    return info, objs

def read_list_file(path):
    """list.txt / trainval.txt / test.txt : 'Image CLASS-ID SPECIES BREED-ID' (#으로 시작하는 줄은 주석)."""
    rows = []
    for line in Path(path).read_text().splitlines():
        if line.startswith("#") or not line.strip():
            continue
        p = line.split()
        rows.append({"image_id": p[0], "class_id": int(p[1]), "species_id": int(p[2])})
    return pd.DataFrame(rows).set_index("image_id")

meta = read_list_file(LIST_FILES["list"]) if "list" in LIST_FILES else pd.DataFrame()
trainval_ids = set(read_list_file(LIST_FILES["trainval"]).index) if "trainval" in LIST_FILES else set()
official_test_ids = set(read_list_file(LIST_FILES["test"]).index) if "test" in LIST_FILES else set()

img_rows, ann_rows = [], []
for iid, img_path in IMG_FILES.items():
    breed = iid.rsplit("_", 1)[0]
    if iid in meta.index:
        species = "cat" if meta.loc[iid, "species_id"] == 1 else "dog"
    else:  # Oxford-IIIT 관례: 고양이 품종 파일명은 대문자로 시작
        species = "cat" if breed[:1].isupper() else "dog"
    has_xml = iid in XML_FILES
    n_obj = 0
    if has_xml:
        info, objs = parse_voc_xml(XML_FILES[iid])
        n_obj = len(objs)
        for o in objs:
            ann_rows.append({"image_id": iid, "breed": breed, "species": species, **info, **o})
    img_rows.append({"image_id": iid, "path": str(img_path), "breed": breed, "species": species,
                     "has_xml": has_xml, "n_obj": n_obj,
                     "in_trainval": iid in trainval_ids, "in_test_list": iid in official_test_ids})
img_df = pd.DataFrame(img_rows).set_index("image_id").sort_index()
ann_df = pd.DataFrame(ann_rows)
ann_df["w"] = ann_df.xmax - ann_df.xmin
ann_df["h"] = ann_df.ymax - ann_df.ymin
ann_df["rel_w"] = ann_df.w / ann_df.xml_w
ann_df["rel_h"] = ann_df.h / ann_df.xml_h
ann_df["rel_area"] = ann_df.rel_w * ann_df.rel_h
ann_df["aspect"] = ann_df.w / ann_df.h
ann_df["cx"] = (ann_df.xmin + ann_df.xmax) / 2 / ann_df.xml_w
ann_df["cy"] = (ann_df.ymin + ann_df.ymax) / 2 / ann_df.xml_h

print("이미지:", len(img_df), "| 박스:", len(ann_df), "| XML 없는 이미지 박스 이름 확인:", ann_df.name.unique())
display(ann_df.head())

### 3-3. 어떤 이미지에 정답(박스)이 있는가? — 공식 분할 목록과의 관계

In [ ]:
cover = pd.DataFrame({
    "이미지 수": [len(img_df), img_df.in_trainval.sum(), img_df.in_test_list.sum()],
    "XML(박스) 보유": [img_df.has_xml.sum(), (img_df.in_trainval & img_df.has_xml).sum(),
                    (img_df.in_test_list & img_df.has_xml).sum()],
}, index=["전체", "공식 trainval.txt", "공식 test.txt"])
cover["보유율"] = cover["XML(박스) 보유"] / cover["이미지 수"].replace(0, np.nan)
display(cover)

**관찰 & 시사점** : Oxford-IIIT Pet 의 얼굴 박스(XML)는 **공식 `trainval` 이미지에만** 제공되고, 공식 `test.txt` 의 이미지에는 박스가 없다(보유율 ≈ 0). 정답 박스가 없으면 IoU·AP·mAP 를 계산할 수 없으므로,

- **정량 평가용 테스트셋**은 **박스가 있는 이미지에서 품종 층화로 따로 떼어 낸(hold-out) 셋**으로 만든다 (4장).
- 공식 `test.txt` 이미지는 **정답 없는 실제 추론 데모**(17장)에 사용한다.
- 만약 사용 중인 데이터 버전에서 test 목록에도 XML 이 충분히 있다면, 4장 코드가 이를 감지해 **공식 test 목록을 그대로 테스트셋으로 사용**한다.

### 3-4. 클래스(종·품종) 분포 — 불균형 확인

In [ ]:
ann_img = img_df[img_df.has_xml]
fig, axes = plt.subplots(1, 3, figsize=(20, 5), gridspec_kw={"width_ratios": [1, 1, 4]})
sp = ann_img.species.value_counts()
axes[0].bar(sp.index, sp.values, color=["#f59f00", "#228be6"]); axes[0].set_title("Annotated images per species")
for i, v in enumerate(sp.values):
    axes[0].text(i, v, f"{v}\n({v/sp.sum():.1%})", ha="center", va="bottom")
nobj = ann_img.n_obj.value_counts().sort_index()
axes[1].bar(nobj.index.astype(str), nobj.values); axes[1].set_title("Boxes per image"); axes[1].set_yscale("log")
br = ann_img.groupby(["breed", "species"]).size().reset_index(name="n").sort_values(["species", "n"])
axes[2].bar(br.breed, br.n, color=br.species.map({"cat": "#f59f00", "dog": "#228be6"}))
axes[2].set_title("Annotated images per breed (orange=cat, blue=dog)"); axes[2].tick_params(axis="x", rotation=80)
plt.tight_layout(); plt.show()

print(ann_df.groupby("species")[["truncated", "difficult"]].mean().rename(columns=lambda c: c + " 비율"))
print("pose 분포:", ann_df.pose.value_counts().to_dict())

**관찰 & 시사점**

1. **종(species) 불균형** : 고양이 12품종, 개 25품종이라 **cat : dog ≈ 1 : 2**. 탐지 모델은 다수 클래스(dog)로 분류가 쏠릴 수 있으므로 11장에서 **클래스 불균형 해소 실험**을 한다. 평가도 전체 mAP 뿐 아니라 **클래스별 AP** 를 함께 본다.
2. **품종 내 분포는 고름** : 품종별 이미지 수는 대체로 비슷하므로(약 100장), 분할 시 **품종 층화(stratify)** 로 각 셋에 모든 품종이 고르게 들어가게 한다.
3. **이미지당 박스 수** : 거의 모든 이미지가 박스 1개(얼굴 1개)이다. → 후처리에서 "이미지당 최대 검출 수"를 줄이는 것이 정밀도(Precision)에 도움이 될 수 있다(15장).
4. `difficult` 박스는 평가에서 **무시(ignore)** 처리한다(PASCAL VOC / COCO 관례 — 맞혀도, 놓쳐도 점수에 영향 없음).

### 3-5. 박스 크기 · 위치 · 종횡비, 이미지 크기

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(19, 10))
sns.histplot(data=ann_df, x="rel_area", hue="species", bins=50, log_scale=True, ax=axes[0, 0], element="step")
axes[0, 0].set_title("Box area / image area (log)")
sns.histplot(data=ann_df, x="aspect", hue="species", bins=50, ax=axes[0, 1], element="step")
axes[0, 1].set_title("Box aspect ratio (w / h)")
sns.scatterplot(data=ann_df.sample(min(len(ann_df), 2000), random_state=0), x="rel_w", y="rel_h",
                hue="species", alpha=.4, s=12, ax=axes[0, 2])
axes[0, 2].set_title("Relative box width vs height")
h2 = axes[1, 0].hist2d(ann_df.cx, ann_df.cy, bins=30, cmap="magma"); axes[1, 0].invert_yaxis()
axes[1, 0].set_title("Box center heatmap (normalized)"); plt.colorbar(h2[3], ax=axes[1, 0])
_imgs = ann_df.drop_duplicates("image_id")
sns.scatterplot(data=_imgs.sample(min(len(_imgs), 2000), random_state=0),
                x="xml_w", y="xml_h", hue="species", alpha=.4, s=12, ax=axes[1, 1])
axes[1, 1].set_title("Image size (px)")
sns.boxplot(data=ann_df, x="species", y="rel_area", ax=axes[1, 2]); axes[1, 2].set_yscale("log")
axes[1, 2].set_title("Relative area by species")
plt.tight_layout(); plt.show()

display(ann_df[["rel_w", "rel_h", "rel_area", "aspect", "xml_w", "xml_h"]].describe(
    percentiles=[.05, .25, .5, .75, .95]).T)

**관찰 & 시사점**

- **박스 크기** : 얼굴은 대체로 이미지 면적의 수 %~수십 % 를 차지하는 **중·대형 객체**지만, 분포의 왼쪽 꼬리(하위 5~25%)에 **이미지의 수 % 이하인 작은 얼굴**(멀리서 찍힌 사진)이 존재한다. 이 구간이 10장 **소형 객체 탐지**의 대상이다. 데이터 특성에 맞게 COCO 의 절대 픽셀 기준(32²/96²) 대신 **상대 면적의 분위수(하위 25% / 상위 25%)로 Small / Medium / Large 를 정의**한다.
- **종횡비** : 1 근처(정사각형에 가까움)에 몰려 있다 → SSD 기본 앵커 종횡비(1, 2, 1/2, 3, 1/3) 중 1·2 계열로 충분히 커버된다.
- **위치** : 중심이 이미지 가운데 위쪽에 몰려 있다(사진 구도). Crop 계열 증강은 얼굴이 가장자리로 가는 상황도 학습시켜 이 편향을 완화한다.
- **이미지 크기** : 대부분 500px 안팎으로 제각각 → 캐시 단계에서 긴 변을 512 로 맞추고, 모델 입력 크기(300/320/512)는 모델 내부 transform 이 맞춘다.

### 3-6. 샘플 이미지 + 정답 박스

In [ ]:
def draw_boxes(ax, img, boxes=(), labels=(), scores=None, color=None, lw=2, title=None):
    """이미지 위에 박스와 라벨(점수)을 그린다. labels 는 1=cat, 2=dog."""
    ax.imshow(img); ax.axis("off")
    for k, (b, l) in enumerate(zip(boxes, labels)):
        c = color or ("#f59f00" if int(l) == 1 else "#228be6")
        ax.add_patch(patches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec=c, lw=lw))
        txt = CLASSES[int(l)] + (f" {scores[k]:.2f}" if scores is not None else "")
        ax.text(b[0], b[1] - 2, txt, color="white", fontsize=8, bbox=dict(fc=c, ec="none", pad=1))
    if title:
        ax.set_title(title, fontsize=9)

fig, axes = plt.subplots(2, 6, figsize=(20, 7))
for r, sp_name in enumerate(["cat", "dog"]):
    ids = ann_img[ann_img.species == sp_name].sample(6, random_state=r).index
    for ax, iid in zip(axes[r], ids):
        img = cv2.cvtColor(cv2.imread(img_df.loc[iid, "path"]), cv2.COLOR_BGR2RGB)
        a = ann_df[ann_df.image_id == iid]
        draw_boxes(ax, img, a[["xmin", "ymin", "xmax", "ymax"]].values, a.name.map(LABEL).values,
                   title=f"{img_df.loc[iid, 'breed']} {img.shape[1]}x{img.shape[0]}")
plt.suptitle("Ground-truth face boxes (orange=cat, blue=dog)"); plt.tight_layout(); plt.show()

**관찰** : 박스가 귀를 포함한 머리 전체를 감싸고, 배경·자세·조명이 다양하다. 개는 품종별로 얼굴 모양 차이가 커서(퍼그 vs 보르조이) 박스 종횡비 분산이 고양이보다 크다.

---
## 4. 데이터 분할 전략

- **정답 박스가 있는 이미지만** 사용한다(이미지 파일이 실제로 존재하고 XML 이 있는 것).
- 공식 test 목록의 XML 보유율이 50% 이상이면 → 공식 test 목록을 테스트셋으로 사용 (데이터 버전 대응)
- 아니면 → 박스 보유 이미지를 **품종 층화**로 **train 70% / val 15% / test 15%** 로 나눈다.
  - **val** : 조기 종료·하이퍼파라미터·후처리 임계값·최종 모델 선택에만 사용
  - **test** : 모든 결정이 끝난 뒤 **마지막에 한 번** 평가 (정보 누수 방지)

In [ ]:
annotated = img_df[img_df.has_xml & (img_df.n_obj > 0)]
if CFG["max_images"]:
    per_breed = max(4, CFG["max_images"] // annotated.breed.nunique())
    annotated = annotated.sample(frac=1, random_state=0).groupby("breed").head(per_breed)

use_official_test = cover.loc["공식 test.txt", "보유율"] >= 0.5
if use_official_test:
    test_ids = sorted(annotated[annotated.in_test_list].index)
    rest = annotated[~annotated.in_test_list]
    print("공식 test 목록에 XML 이 충분 → 공식 test 를 테스트셋으로 사용")
else:
    rest, test_part = train_test_split(annotated, test_size=CFG["test_size"], stratify=annotated.breed,
                                       random_state=CFG["seed"])
    test_ids = sorted(test_part.index)
    print("공식 test 목록에는 정답 박스가 없음 → 박스 보유 이미지에서 품종 층화 hold-out 테스트셋 구성")
tr_part, va_part = train_test_split(rest, test_size=CFG["val_size"] / (1 - CFG["test_size"]),
                                    stratify=rest.breed, random_state=CFG["seed"])
train_ids, val_ids = sorted(tr_part.index), sorted(va_part.index)
assert not (set(train_ids) & set(val_ids)) and not (set(train_ids) & set(test_ids)) and not (set(val_ids) & set(test_ids))

# 정답이 없는 공식 test 이미지 (17장 추론 데모용)
demo_ids = sorted(img_df[img_df.in_test_list & ~img_df.has_xml].index)

split_summary = pd.DataFrame({
    n: {"images": len(ids), "boxes": int(img_df.loc[ids, "n_obj"].sum()),
        "cat": int((img_df.loc[ids, "species"] == "cat").sum()), "dog": int((img_df.loc[ids, "species"] == "dog").sum()),
        "cat 비율": (img_df.loc[ids, "species"] == "cat").mean(), "breeds": img_df.loc[ids, "breed"].nunique()}
    for n, ids in [("train", train_ids), ("val", val_ids), ("test", test_ids)]}).T
display(split_summary)
print("정답 없는 공식 test 이미지(추론 데모용):", len(demo_ids))

**결과 해석** : 세 셋 모두 고양이 비율(약 1/3)과 품종 수(37)가 같게 유지된다. 층화 덕분에 val·test 에서 특정 품종이 빠져 지표가 왜곡되는 일을 막았다. 세 셋 사이 이미지 중복은 0 이다.

---
## 5. 데이터 로드 · 전처리 · 증강

### 5-1. 메모리 캐시 + 박스 좌표 변환
- 이미지를 RGB 로 한 번만 읽어 **긴 변 512px 로 축소(종횡비 유지)** 해 메모리에 둔다(약 2GB). 매 epoch JPEG 디코딩이 사라져 여러 실험을 빠르게 반복할 수 있다.
- 이미지를 줄인 비율(scale)만큼 **박스 좌표도 같이 변환**한다. XML 의 `<size>` 와 실제 이미지 크기가 다른 드문 경우에도 실제 크기 기준으로 보정하고, 이미지 밖으로 나간 좌표는 잘라낸다(clip).
- 흑백·CMYK·PNG 가 섞인 파일도 `cv2.IMREAD_COLOR` 로 3채널로 통일한다. 읽을 수 없는 파일은 제외한다.

In [ ]:
def load_sample(iid, max_side=CFG["cache_max_side"]):
    """이미지 1장을 읽어 축소하고, 박스를 같은 비율로 변환한다.

    Returns: dict(img=RGB uint8 HxWx3, boxes=float32 Nx4 (xyxy, 축소 좌표), labels=int64 N,
                  difficult=bool N, orig_size=(H, W)) 또는 읽기 실패 시 None
    """
    bgr = cv2.imread(img_df.loc[iid, "path"], cv2.IMREAD_COLOR)
    if bgr is None:
        return None
    H, W = bgr.shape[:2]
    s = min(1.0, max_side / max(H, W))
    img = cv2.cvtColor(cv2.resize(bgr, (round(W * s), round(H * s)), interpolation=cv2.INTER_AREA) if s < 1 else bgr,
                       cv2.COLOR_BGR2RGB)
    info, objs = parse_voc_xml(XML_FILES[iid]) if iid in XML_FILES else ({"xml_w": W, "xml_h": H}, [])
    sx = W / info["xml_w"] if info["xml_w"] else 1.0
    sy = H / info["xml_h"] if info["xml_h"] else 1.0
    boxes = np.array([[o["xmin"] * sx, o["ymin"] * sy, o["xmax"] * sx, o["ymax"] * sy] for o in objs],
                     dtype=np.float32).reshape(-1, 4) * s
    h, w = img.shape[:2]
    boxes[:, [0, 2]] = boxes[:, [0, 2]].clip(0, w); boxes[:, [1, 3]] = boxes[:, [1, 3]].clip(0, h)
    keep = (boxes[:, 2] - boxes[:, 0] > 1) & (boxes[:, 3] - boxes[:, 1] > 1)
    labels = np.array([LABEL.get(o["name"], LABEL[img_df.loc[iid, "species"]]) for o in objs], dtype=np.int64)
    diff = np.array([o["difficult"] for o in objs], dtype=bool)
    return {"img": img, "boxes": boxes[keep], "labels": labels[keep], "difficult": diff[keep], "orig_size": (H, W)}

t0 = time.time()
all_ids = train_ids + val_ids + test_ids
with ThreadPoolExecutor(max_workers=8) as ex:
    CACHE = dict(zip(all_ids, ex.map(load_sample, all_ids)))
bad = [k for k, v in CACHE.items() if v is None or len(v["boxes"]) == 0]
for k in bad:
    CACHE.pop(k)
train_ids, val_ids, test_ids = ([i for i in ids if i in CACHE] for ids in (train_ids, val_ids, test_ids))
mem = sum(v["img"].nbytes for v in CACHE.values()) / 1e9
print(f"캐시 완료 {time.time()-t0:.1f}s | {len(CACHE)}장, {mem:.2f} GB | 제외(읽기 실패/박스 없음): {len(bad)}")
print(f"train {len(train_ids)} / val {len(val_ids)} / test {len(test_ids)}")

# 평가용 정답(GT) 사전 : 캐시 좌표계 기준
GT = {iid: {"boxes": v["boxes"], "labels": v["labels"], "difficult": v["difficult"],
            "size": v["img"].shape[:2]} for iid, v in CACHE.items()}

**결과 해석** : 축소 이미지와 박스가 같은 좌표계에 있으므로 이후 학습·예측·평가가 모두 **캐시 좌표계**로 통일된다(상대 면적 기반 지표는 축소 비율과 무관).


### 5-2. 크기 구간(Small / Medium / Large) 정의
train 박스의 상대 면적 분위수로 경계를 정한다. 이 경계는 모든 셋·모델의 **크기별 AP** 계산에 동일하게 쓰인다.

In [ ]:
def rel_area(boxes, size):
    """박스 면적 / 이미지 면적 (size = (H, W))."""
    boxes = np.asarray(boxes).reshape(-1, 4)
    return (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]) / (size[0] * size[1])

train_areas = np.concatenate([rel_area(GT[i]["boxes"], GT[i]["size"]) for i in train_ids])
AREA_THR = tuple(np.quantile(train_areas, [0.25, 0.75]).round(4))
AREA_RANGES = {"S": (0.0, AREA_THR[0]), "M": AREA_THR, "L": (AREA_THR[1], 1.01)}
print(f"Small < {AREA_THR[0]:.3f} ≤ Medium < {AREA_THR[1]:.3f} ≤ Large  (이미지 면적 대비)")
print(f"  ≈ 정사각형 박스 한 변 기준: {np.sqrt(AREA_THR[0]):.1%} / {np.sqrt(AREA_THR[1]):.1%} of image side")
for nm, ids in [("val", val_ids), ("test", test_ids)]:
    a = np.concatenate([rel_area(GT[i]["boxes"], GT[i]["size"]) for i in ids])
    print(f"  {nm:5s} S/M/L 박스 수:", [int(((a >= lo) & (a < hi)).sum()) for lo, hi in AREA_RANGES.values()])

### 5-3. Dataset 클래스와 증강 정책

`torchvision.transforms.v2` 는 이미지와 **Bounding Box 를 함께 변환**해 준다(뒤집기·자르기 시 박스 좌표 자동 보정, 이미지 밖으로 나간 박스는 `SanitizeBoundingBoxes` 가 제거).

| 정책 | 구성 | 의도 |
|---|---|---|
| `basic` | 좌우 반전 | 최소 증강 (Baseline) |
| `ssd` | **PhotometricDistort**(밝기·대비·채도·색상) + **ZoomOut**(캔버스 1~2.5배 확대 후 배치 → 객체가 작아짐) + **RandomIoUCrop**(박스와 IoU 조건을 만족하는 랜덤 crop → 객체가 커지거나 잘림) + 좌우 반전 | SSD 논문의 표준 증강. 크기·위치·색 다양성 ↑ |
| `ssd` + **Mosaic** | 4장을 2×2 로 합성(확률 0.5) 후 `ssd` 증강 | 한 장에 여러 객체·작은 객체·다양한 배경 → 소형 객체·문맥 일반화 (YOLO 계열 핵심 증강) |

- 미션 6(X-ray)과 달리 **좌우 반전은 사용**한다. 동물 얼굴은 좌우 대칭이고 반전 영상도 현실에 존재한다. 상하 반전은 현실에 없는 자세라 쓰지 않는다.
- 정규화(ImageNet/COCO 평균)와 입력 크기 조정(300/320/512)은 torchvision 탐지 모델 **내부 transform** 이 수행하므로, Dataset 은 0~1 float 텐서만 넘긴다.

In [ ]:
class PetFaceDataset(Dataset):
    """캐시된 (이미지, 박스, 라벨)을 v2 transform 에 넣어 (image_tensor, target_dict)를 반환한다.

    Args:
        ids       : 사용할 image_id 리스트
        transform : torchvision.transforms.v2 파이프라인 (이미지·박스를 함께 변환)
        mosaic_p  : 4장 Mosaic 합성 확률 (0 이면 사용 안 함)
    """
    def __init__(self, ids, transform, mosaic_p=0.0, mosaic_size=512):
        self.ids, self.transform, self.mosaic_p, self.S = list(ids), transform, mosaic_p, mosaic_size

    def __len__(self):
        return len(self.ids)

    def _raw(self, i):
        c = CACHE[self.ids[i]]
        return c["img"], c["boxes"], c["labels"]

    def _mosaic(self, i):
        """중심점(cx, cy)을 기준으로 4장을 사분면에 비율 유지 축소해 배치한다 (박스는 잘리지 않음)."""
        S = self.S
        cx, cy = int(random.uniform(0.3, 0.7) * S), int(random.uniform(0.3, 0.7) * S)
        canvas = np.full((S, S, 3), (124, 116, 104), np.uint8)
        idxs = [i] + random.sample(range(len(self.ids)), 3)
        quads = [(0, 0, cx, cy), (cx, 0, S, cy), (0, cy, cx, S), (cx, cy, S, S)]   # x0,y0,x1,y1
        all_b, all_l = [], []
        for k, (x0, y0, x1, y1) in zip(idxs, quads):
            img, b, l = self._raw(k)
            qw, qh = x1 - x0, y1 - y0
            if qw < 8 or qh < 8:
                continue
            s = min(qw / img.shape[1], qh / img.shape[0])
            nw, nh = max(1, int(img.shape[1] * s)), max(1, int(img.shape[0] * s))
            ox = x1 - nw if x0 == 0 else x0          # 중심점 쪽으로 붙여서 배치
            oy = y1 - nh if y0 == 0 else y0
            canvas[oy:oy + nh, ox:ox + nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
            all_b.append(b * s + np.array([ox, oy, ox, oy], np.float32)); all_l.append(l)
        return canvas, np.concatenate(all_b), np.concatenate(all_l)

    def __getitem__(self, i):
        if self.mosaic_p > 0 and random.random() < self.mosaic_p:
            img, boxes, labels = self._mosaic(i)
        else:
            img, boxes, labels = self._raw(i)
        image = tv_tensors.Image(torch.from_numpy(np.ascontiguousarray(img)).permute(2, 0, 1))
        target = {"boxes": tv_tensors.BoundingBoxes(torch.as_tensor(boxes, dtype=torch.float32), format="XYXY",
                                                    canvas_size=tuple(image.shape[-2:])),
                  "labels": torch.as_tensor(labels, dtype=torch.int64)}
        image, target = self.transform(image, target)
        return image, target

def collate_fn(batch):
    """탐지 모델은 이미지마다 박스 수가 다르므로 텐서로 쌓지 않고 리스트로 묶는다."""
    return tuple(zip(*batch))

FILL = defaultdict(lambda: 0, {tv_tensors.Image: (124, 116, 104)})
_to_float = [T2.ToDtype(torch.float32, scale=True)]
AUGS = {
    "basic": T2.Compose([T2.RandomHorizontalFlip(0.5)] + _to_float + [T2.SanitizeBoundingBoxes()]),
    "ssd": T2.Compose([
        T2.RandomPhotometricDistort(p=0.5),
        T2.RandomZoomOut(fill=FILL, side_range=(1.0, 2.5), p=0.4),
        T2.RandomIoUCrop(),
        T2.RandomHorizontalFlip(0.5),
    ] + _to_float + [T2.SanitizeBoundingBoxes()]),
}
eval_tf = T2.Compose(_to_float)

def make_loader(ids, tf=eval_tf, shuffle=False, mosaic_p=0.0, sampler=None, batch_size=None):
    """PetFaceDataset → DataLoader. sampler 가 주어지면 shuffle 대신 sampler 를 사용."""
    ds = PetFaceDataset(ids, tf, mosaic_p=mosaic_p)
    return DataLoader(ds, batch_size=batch_size or CFG["batch_size"], shuffle=shuffle and sampler is None,
                      sampler=sampler, num_workers=CFG["num_workers"], collate_fn=collate_fn,
                      pin_memory=device.type == "cuda", persistent_workers=CFG["num_workers"] > 0)

# 증강 결과 시각화 (같은 이미지에 각 정책을 6번 적용)
vis_id = train_ids[0]
fig, axes = plt.subplots(3, 6, figsize=(20, 10))
for r, (nm, tf, mp) in enumerate([("basic", AUGS["basic"], 0), ("ssd", AUGS["ssd"], 0), ("ssd + mosaic", AUGS["ssd"], 1.0)]):
    ds = PetFaceDataset(train_ids, tf, mosaic_p=mp)
    for c in range(6):
        random.seed(r * 10 + c); torch.manual_seed(r * 10 + c)
        im, tg = ds[0]
        draw_boxes(axes[r, c], im.permute(1, 2, 0).numpy(), tg["boxes"].numpy(), tg["labels"].numpy(),
                   title=f"{nm} {tuple(im.shape[-2:])}" if c == 0 else None)
plt.tight_layout(); plt.show()
seed_everything()

**결과 해석** : `ssd` 정책에서는 같은 사진이 확대(ZoomOut → 얼굴이 작아짐)·잘림(IoUCrop → 얼굴이 커짐)·색 변화를 거치며 매번 다르게 보이고, 박스가 이미지와 함께 정확히 따라 움직인다. Mosaic 은 한 장에 4개의 얼굴을 작게 담아 **작은 객체·다중 객체 학습 신호**를 크게 늘린다.



---
## 6. 평가 지표 구현 — IoU · Precision / Recall · AP · mAP

| 지표 | 정의 | 이 문제에서의 의미 |
|---|---|---|
| **IoU** | 예측 박스 ∩ 정답 박스 / 합집합 | 위치 정확도. IoU ≥ 임계값이면 같은 클래스 정답과 "매칭"(TP) |
| **Precision** | TP / (TP + FP) | 모델이 "얼굴"이라 한 것 중 진짜 얼굴(+맞는 종) 비율 → 오탐이 적은가 |
| **Recall** | TP / (TP + FN) | 실제 얼굴 중 찾아낸 비율 → 놓친 얼굴이 적은가 |
| **AP** | Precision–Recall 곡선 아래 면적 (COCO 101-point 보간) | **점수 임계값과 무관한** 클래스별 탐지 성능 |
| **mAP@0.5** | IoU 0.5 기준 AP 의 클래스 평균 | PASCAL VOC 지표 — "대략 맞게 찾았나" |
| **mAP@0.75** | IoU 0.75 기준 | 엄격한 위치 정확도 |
| **mAP@[.5:.95]** | IoU 0.5~0.95(0.05 간격) 10개 평균 | **COCO 주 지표** — 검출 + 정밀한 위치를 모두 반영. 모델 선택 기준 |
| AP_S / AP_M / AP_L | 크기 구간별 mAP@[.5:.95] | 소형 객체 성능 |

**매칭 규칙 (COCO 방식)** : 이미지·클래스별로 예측을 점수 내림차순으로 보며, 아직 매칭되지 않은 정답 중 IoU 가 가장 큰 것과 매칭한다. 한 정답에는 한 예측만 TP 이고 나머지 중복 예측은 FP 이다. `difficult` 또는 평가 크기 구간 밖의 정답은 **ignore** (그 정답과 매칭된 예측은 TP 도 FP 도 아님).

**운영점(Operating point) 지표** : AP 는 모든 점수 임계값을 아우르지만, 실제 서비스는 하나의 임계값(`CFG["op_score"]=0.5`)을 쓴다. 그 임계값에서 IoU 0.5 기준 **Precision · Recall · F1 · 평균 IoU** 와, 이미지마다 최고 점수 박스의 종(cat/dog)이 맞았는지(**Top-1 종 정확도**)도 함께 계산한다.

> 외부 라이브러리(pycocotools 등) 없이 직접 구현해 계산 과정을 투명하게 보이고, 같은 함수로 SSD 와 YOLO 를 **동일 기준**으로 평가한다.

In [ ]:
def iou_np(a, b):
    """(N,4) x (M,4) xyxy 박스 IoU 행렬 (numpy)."""
    a, b = np.asarray(a, np.float64).reshape(-1, 4), np.asarray(b, np.float64).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    lt = np.maximum(a[:, None, :2], b[None, :, :2]); rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.prod(np.clip(rb - lt, 0, None), axis=2)
    area = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-12)

def coco_ap(tp, fp, n_pos):
    """점수순 정렬된 TP/FP(0/1) 배열 → (AP, recall 배열, precision 배열). COCO 101-point 보간."""
    if n_pos == 0:
        return np.nan, np.array([]), np.array([])
    tpc, fpc = np.cumsum(tp), np.cumsum(fp)
    rec = tpc / n_pos
    prec = tpc / np.maximum(tpc + fpc, 1e-12)
    env = np.maximum.accumulate(prec[::-1])[::-1] if len(prec) else prec      # 우측 최대값 포락선
    rs = np.linspace(0, 1, 101)
    idx = np.searchsorted(rec, rs, side="left")
    q = np.array([env[i] if i < len(env) else 0.0 for i in idx])
    return q.mean(), rec, prec

def _match_image(pb, ps, pa, gb, gign, ious, thr):
    """한 이미지·한 클래스·한 IoU 임계값에서 COCO 방식 greedy 매칭.
    Returns: tp(bool D), ign(bool D), matched_iou(float D)"""
    D, G = len(pb), len(gb)
    tp, ign, miou = np.zeros(D, bool), np.zeros(D, bool), np.zeros(D)
    used = np.zeros(G, bool)
    for d in range(D):
        best, best_iou = -1, min(thr, 1 - 1e-10)
        for g in range(G):
            if used[g]:
                continue
            if best > -1 and not gign[best] and gign[g]:   # 이미 일반 GT 와 매칭 후보가 있으면 ignore GT 는 보지 않음
                break
            if ious[d, g] < best_iou:
                continue
            best, best_iou = g, ious[d, g]
        if best == -1:
            continue
        used[best] = True
        tp[d], ign[d], miou[d] = True, gign[best], ious[d, best]
    return tp, ign, miou

def evaluate_detections(preds, ids, area_range=None, iou_thrs=IOU_THRS, op_score=None, max_dets=100):
    """예측 dict {image_id: {'boxes','scores','labels'}} 을 GT 와 비교해 지표를 계산한다.

    Returns: dict
      - 'AP'       : {class_id: AP 배열(len(iou_thrs))}
      - 'mAP', 'mAP50', 'mAP75', 'AP_cat', 'AP_dog'
      - 'pr'       : {class_id: (recall, precision)} @IoU 0.5 (PR 곡선용)
      - 운영점 지표 (area_range 가 None 일 때): precision, recall, f1, mean_iou, TP, FP, FN, top1_acc
    """
    op_score = CFG["op_score"] if op_score is None else op_score
    T = len(iou_thrs)
    res = {"AP": {}, "pr": {}}
    op = {"TP": 0, "FP": 0, "FN": 0, "ious": []}
    for c in CLASS_IDS:
        scores, tps, igns, mious = [], [[] for _ in range(T)], [[] for _ in range(T)], []
        n_pos = 0
        for iid in ids:
            g, p = GT[iid], preds.get(iid, {"boxes": np.zeros((0, 4)), "scores": np.zeros(0), "labels": np.zeros(0)})
            gm = g["labels"] == c
            gb, gign = g["boxes"][gm], g["difficult"][gm].copy()
            if area_range is not None:
                ga = rel_area(gb, g["size"]); gign |= (ga < area_range[0]) | (ga >= area_range[1])
            go = np.argsort(gign, kind="stable"); gb, gign = gb[go], gign[go]    # 일반 GT 먼저
            n_pos += int((~gign).sum())
            pm = np.asarray(p["labels"]) == c
            pb, ps = np.asarray(p["boxes"])[pm].reshape(-1, 4), np.asarray(p["scores"])[pm]
            o = np.argsort(-ps, kind="stable")[:max_dets]; pb, ps = pb[o], ps[o]
            ious = iou_np(pb, gb)
            pa = rel_area(pb, g["size"]) if len(pb) else np.zeros(0)
            for t, thr in enumerate(iou_thrs):
                tp, ign, miou = _match_image(pb, ps, pa, gb, gign, ious, thr)
                if area_range is not None:   # 매칭되지 않은 구간 밖 예측은 ignore
                    ign |= (~tp) & ((pa < area_range[0]) | (pa >= area_range[1]))
                tps[t].append(tp); igns[t].append(ign)
                if t == 0:
                    mious.append(miou)
            scores.append(ps)
        sc = np.concatenate(scores) if scores else np.zeros(0)
        order = np.argsort(-sc, kind="stable")
        ap = np.zeros(T)
        for t in range(T):
            tp = np.concatenate(tps[t])[order]; ig = np.concatenate(igns[t])[order]
            ap[t], rec, prec = coco_ap((tp & ~ig).astype(float), (~tp & ~ig).astype(float), n_pos)
            if t == 0:
                res["pr"][c] = (rec, prec)
                if area_range is None:          # 운영점 지표 @IoU0.5, score>=op_score
                    keep = (sc[order] >= op_score) & ~ig
                    tp_op = int((tp & keep).sum())
                    op["TP"] += tp_op; op["FP"] += int((~tp & keep).sum()); op["FN"] += n_pos - tp_op
                    mi = np.concatenate(mious)[order]
                    op["ious"].append(mi[tp & keep])
        res["AP"][c] = ap
    APs = np.array([res["AP"][c] for c in CLASS_IDS])        # (C, T)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        res["mAP"] = float(np.nanmean(APs)); res["mAP50"] = float(np.nanmean(APs[:, 0]))
        res["mAP75"] = float(np.nanmean(APs[:, list(iou_thrs).index(0.75)]))
        res["AP_cat"], res["AP_dog"] = float(np.nanmean(APs[0])), float(np.nanmean(APs[1]))
        res["AP50_cat"], res["AP50_dog"] = float(APs[0, 0]), float(APs[1, 0])
    if area_range is None:
        TP, FP, FN = op["TP"], op["FP"], op["FN"]
        res.update(TP=TP, FP=FP, FN=FN, precision=TP / max(TP + FP, 1), recall=TP / max(TP + FN, 1))
        res["f1"] = 2 * res["precision"] * res["recall"] / max(res["precision"] + res["recall"], 1e-12)
        ious = np.concatenate(op["ious"]) if op["ious"] else np.zeros(0)
        res["mean_iou"] = float(ious.mean()) if len(ious) else np.nan
        # Top-1 종 정확도: 이미지별 최고 점수 박스의 라벨 == 정답 종
        hit = [len(preds.get(i, {}).get("scores", [])) > 0 and
               int(np.asarray(preds[i]["labels"])[np.argmax(preds[i]["scores"])]) == int(GT[i]["labels"][0]) for i in ids]
        res["top1_acc"] = float(np.mean(hit))
    return res

def full_metrics(preds, ids):
    """전체 지표 + 크기별(S/M/L) mAP@[.5:.95] 를 한 번에 계산."""
    m = evaluate_detections(preds, ids)
    for k, rng in AREA_RANGES.items():
        m[f"AP_{k}"] = evaluate_detections(preds, ids, area_range=rng)["mAP"]
    return m

METRIC_COLS = ["mAP", "mAP50", "mAP75", "AP_cat", "AP_dog", "AP_S", "AP_M", "AP_L",
               "precision", "recall", "f1", "mean_iou", "top1_acc"]

# --- 간단한 자체 검증: 정답을 그대로 예측으로 넣으면 모든 지표 = 1 이어야 한다 ---
_perfect = {i: {"boxes": GT[i]["boxes"], "scores": np.ones(len(GT[i]["boxes"])), "labels": GT[i]["labels"]} for i in val_ids}
_m = full_metrics(_perfect, val_ids)
print("정답=예측 검증:", {k: round(_m[k], 3) for k in ["mAP", "mAP50", "precision", "recall", "mean_iou", "top1_acc"]})
_shift = {i: {"boxes": GT[i]["boxes"] + np.array([5, 5, 5, 5], np.float32), "scores": np.ones(len(GT[i]["boxes"])),
              "labels": GT[i]["labels"]} for i in val_ids}
print("5px 이동 박스 :", {k: round(evaluate_detections(_shift, val_ids)[k], 3) for k in ["mAP50", "mAP75", "mAP", "mean_iou"]})

**결과 해석 (평가 함수 검증)** : 정답을 그대로 예측으로 넣으면 모든 지표가 1.0 이 되어 구현이 맞음을 확인한다. 박스를 5px 만 밀어도 IoU 0.5 기준(mAP50)은 거의 그대로지만 **엄격한 임계값(0.75~0.95)이 포함된 mAP@[.5:.95] 는 떨어진다** — 이것이 mAP@[.5:.95] 가 "위치 정밀도"까지 반영하는 지표인 이유다.

---
## 7. 모델 · 학습 공통 유틸

### 7-1. SSD 구조 요약과 모델 생성 함수
**SSD(Single Shot MultiBox Detector)** 는 한 번의 순전파로 여러 해상도의 특징맵(VGG16 기준 38×38 → 1×1, 6단계)마다 미리 정한 **기본 박스(default box, anchor)** 에 대해 (1) 클래스 점수, (2) 박스 위치 보정값을 예측한다. 큰 특징맵은 작은 객체, 작은 특징맵은 큰 객체를 담당한다.

- 학습 : 정답과 IoU ≥ 0.5 인 앵커를 양성으로 매칭 → **Smooth-L1 (박스) + Cross-Entropy (클래스) + Hard Negative Mining(음성:양성 = 3:1)**
- 추론 : 클래스별 점수 필터 → 상위 k 개 → **NMS** 로 중복 제거

`build_detector()` 하나로 모든 SSD 변형을 만든다.

| arch | 백본 | 입력 | 앵커 | 비고 |
|---|---|---|---|---|
| `ssd300` | VGG16 | 300 | SSD300 기본 (scale 0.07~1.05) | COCO 사전학습, 기준 모델 |
| `ssd512` | VGG16 | **512** | **SSD512 논문 scale (0.04~0.9, 더 작음)** | 소형 객체 개선 (10장) |
| `ssdlite` | **MobileNetV3-Large** | 320 | SSDLite 기본 | 경량 모델 (12장, 16장) |

COCO 사전학습 모델의 **분류 헤드(91클래스)만 3클래스(배경·cat·dog)로 교체**하고, 박스 회귀 헤드·백본은 COCO 가중치를 그대로 이어받는다(COCO 에도 cat, dog 클래스가 있어 유용한 특징을 이미 학습했다).

In [ ]:
SSD300_SCALES = [0.07, 0.15, 0.33, 0.51, 0.69, 0.87, 1.05]   # torchvision ssd300_vgg16 기본
SSD512_SCALES = [0.04, 0.10, 0.26, 0.42, 0.58, 0.74, 0.90]   # SSD512 논문 설정 (더 작은 앵커)
SSD_RATIOS = [[2], [2, 3], [2, 3], [2, 3], [2], [2]]

def _last_conv(module):
    """모듈 안의 마지막 Conv2d (SSDLite 헤드는 depthwise-conv 블록 뒤에 1x1 conv 가 붙음)."""
    return [m for m in module.modules() if isinstance(m, nn.Conv2d)][-1]

def init_background_prior(cls_head, num_classes=NUM_CLASSES, prior=0.01):
    """Focal loss 용 초기화: 학습 시작 시 '전경일 확률' 이 prior 가 되도록 배경 logit bias 를 크게 둔다.
    (RetinaNet 의 prior-probability 초기화를 softmax 버전으로 옮긴 것: 초반 손실 폭주 방지)"""
    b_bg = math.log((1 - prior) / prior * (num_classes - 1))
    for m in cls_head.module_list:
        conv = _last_conv(m)
        with torch.no_grad():
            conv.bias.zero_(); conv.bias[0::num_classes] = b_bg

def build_detector(arch="ssd300", pretrained=None, loss="ce"):
    """COCO 사전학습 SSD 를 만들고 분류 헤드를 3클래스로 교체한다.

    Args:
        arch : 'ssd300' | 'ssd512' | 'ssdlite'
        loss : 'ce' (기본 SSD 손실) | 'focal_giou' (12장 개선 손실)
    Returns: (model, param_groups)  — head 는 lr_head, 나머지는 lr_backbone
    """
    pretrained = PRETRAINED if pretrained is None else pretrained
    if arch in ("ssd300", "ssd512"):
        m = ssd300_vgg16(weights=SSD300_VGG16_Weights.COCO_V1 if pretrained else None, weights_backbone=None)
        size = 300
        if arch == "ssd512":
            size = 512
            m.anchor_generator = DefaultBoxGenerator(SSD_RATIOS, scales=SSD512_SCALES, steps=None)
            m.transform = GeneralizedRCNNTransform(min_size=size, max_size=size, image_mean=m.transform.image_mean,
                                                   image_std=m.transform.image_std, size_divisible=1,
                                                   fixed_size=(size, size))
        in_ch = det_utils.retrieve_out_channels(m.backbone, (size, size))
        m.head.classification_head = SSDClassificationHead(in_ch, m.anchor_generator.num_anchors_per_location(), NUM_CLASSES)
    elif arch == "ssdlite":
        m = ssdlite320_mobilenet_v3_large(weights=SSDLite320_MobileNet_V3_Large_Weights.COCO_V1 if pretrained else None,
                                          weights_backbone=None)
        in_ch = det_utils.retrieve_out_channels(m.backbone, (320, 320))
        m.head.classification_head = SSDLiteClassificationHead(
            in_ch, m.anchor_generator.num_anchors_per_location(), NUM_CLASSES,
            partial(nn.BatchNorm2d, eps=0.001, momentum=0.03))
    else:
        raise ValueError(arch)
    # 평가용 후처리 설정을 모든 모델에서 동일하게 (mAP 계산은 낮은 점수까지 포함해야 정확)
    m.score_thresh, m.nms_thresh, m.detections_per_img, m.topk_candidates = 0.01, 0.45, 100, 400
    if loss == "focal_giou":
        init_background_prior(m.head.classification_head)
        m.compute_loss = partial(focal_giou_loss, m)       # 인스턴스 메서드 교체 (forward 가 self.compute_loss 호출)
    head_params = list(m.head.classification_head.parameters())
    head_ids = {id(p) for p in head_params}
    rest = [p for p in m.parameters() if p.requires_grad and id(p) not in head_ids]
    groups = [{"params": head_params, "lr": CFG["lr_head"]}, {"params": rest, "lr": CFG["lr_backbone"]}]
    return m.to(device), groups

def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trn, tot

### 7-2. 개선 손실함수 — Focal Loss + GIoU Loss (12장에서 사용)

기본 SSD 손실의 한계와 대안:

| 구성 | 기본 SSD | 개선 | 이유 |
|---|---|---|---|
| 분류 | CE + **Hard Negative Mining** (어려운 음성 앵커만 3:1 로 골라 학습) | **Softmax Focal Loss** (γ=2, 모든 앵커 사용) | 쉬운 배경 앵커의 손실을 (1−p)^γ 로 자동 감쇠 → 휴리스틱 없이 어려운 예제(작은 얼굴·헷갈리는 종)에 집중. 소수 클래스(cat) 오분류에도 더 큰 가중치 |
| 박스 | Smooth-L1 (오프셋 좌표 회귀) | Smooth-L1 **+ GIoU Loss** | 평가 지표(IoU)를 직접 최적화. 좌표별 독립 회귀의 한계를 보완해 **IoU 0.75+ 구간(mAP75)** 개선 기대 |

torchvision 의 `SSD.forward` 는 `self.compute_loss(...)` 를 호출하므로, 같은 인터페이스의 함수를 인스턴스에 바인딩해 손실만 교체한다(나머지 매칭·추론 로직은 그대로).

In [ ]:
def focal_giou_loss(self, targets, head_outputs, anchors, matched_idxs, gamma=2.0, giou_weight=1.0):
    """torchvision SSD.compute_loss 대체 함수: Softmax Focal (분류) + SmoothL1 + GIoU (박스).

    Args: torchvision SSD.compute_loss 와 동일
        targets      : [{'boxes': (G,4), 'labels': (G,)}] — 모델 입력 크기 좌표
        head_outputs : {'bbox_regression': (B,A,4), 'cls_logits': (B,A,C)}
        anchors      : [(A,4)] 이미지별 default box
        matched_idxs : [(A,)] 앵커별 매칭된 GT 인덱스 (-1: 배경)
    Returns: {'bbox_regression': loss, 'classification': loss}  (양성 앵커 수로 정규화)
    """
    bbox_reg, cls_logits = head_outputs["bbox_regression"], head_outputs["cls_logits"]
    num_fg, l1_terms, giou_terms, cls_targets = 0, [], [], []
    for t, br, an, mi in zip(targets, bbox_reg, anchors, matched_idxs):
        fg = torch.where(mi >= 0)[0]
        gt_idx = mi[fg]
        num_fg += fg.numel()
        gt_boxes = t["boxes"][gt_idx].float()
        br_fg, an_fg = br[fg].float(), an[fg].float()
        target_reg = self.box_coder.encode_single(gt_boxes, an_fg)
        l1_terms.append(F.smooth_l1_loss(br_fg, target_reg, reduction="sum"))
        if fg.numel() > 0:
            pred_boxes = self.box_coder.decode_single(br_fg, an_fg)
            giou_terms.append(generalized_box_iou_loss(pred_boxes, gt_boxes, reduction="sum"))
        else:
            giou_terms.append(br.sum() * 0.0)
        ct = torch.zeros(cls_logits.size(1), dtype=torch.int64, device=cls_logits.device)
        ct[fg] = t["labels"][gt_idx]
        cls_targets.append(ct)
    cls_targets = torch.stack(cls_targets).view(-1)
    C = cls_logits.size(-1)
    logp = F.log_softmax(cls_logits.float().view(-1, C), dim=-1)
    logpt = logp.gather(1, cls_targets[:, None]).squeeze(1)
    focal = -((1 - logpt.exp()) ** gamma) * logpt
    N = max(1, num_fg)
    return {"bbox_regression": (torch.stack(l1_terms).sum() + giou_weight * torch.stack(giou_terms).sum()) / N,
            "classification": focal.sum() / N}

for a in ["ssd300", "ssd512", "ssdlite"]:
    _m, _ = build_detector(a, pretrained=False)
    trn, tot = count_params(_m)
    print(f"{a:8s}: 파라미터 {tot/1e6:6.2f}M (학습 {trn/1e6:6.2f}M) | 앵커/위치 {_m.anchor_generator.num_anchors_per_location()}")
del _m

**결과 해석** : SSD300-VGG16 은 약 2,400만 파라미터, SSDLite-MobileNetV3 는 약 300만으로 **8배가량 가볍다**. SSD512 는 같은 네트워크에 입력·앵커만 바꾼 것이라 파라미터 수가 같지만, 특징맵이 커져 연산량(FLOPs)은 약 3배가 된다(16장에서 측정). 사전학습 시 VGG 앞단 일부 층은 torchvision 기본값대로 고정(frozen)되어 있어 "학습" 파라미터가 전체보다 조금 적다.



### 7-3. 학습 루프 · 검증 · 조기 종료 · 과적합 진단

- **AdamW + Discriminative LR**, 1 epoch **선형 warm-up** 후 **cosine 감소** (iteration 단위) : 무작위 초기화된 새 헤드의 큰 gradient 가 사전학습 가중치를 망가뜨리는 것을 막는다.
- **Gradient clipping(10)** : 탐지 손실은 초반에 튈 수 있어 안정화.
- 매 epoch : train 손실(박스·분류), **val 손실**(BN 은 eval 고정), **val mAP@[.5:.95] / mAP50** 기록.
- **조기 종료 · 최적 가중치 선택 기준 = val mAP@[.5:.95]** (탐지에서는 손실보다 최종 지표가 목적에 직접 맞음).
- 종료 후 **증강 없는 train 일부(최대 600장)** 와 val 을 다시 평가해 공정한 train/val 격차를 구한다.

**과적합/과소적합 판정 규칙** (`diagnose`, 경험적 기준 — 학습곡선과 함께 해석)
- **과소적합** : train mAP50 < 0.80 → 학습 데이터조차 충분히 맞히지 못함
- **과적합** : train−val mAP@[.5:.95] 격차 > 0.10, 또는 best epoch 이후 val loss 가 15% 이상 재상승
- 그 외 **적정 적합**

In [ ]:
def _to_device(images, targets=None):
    images = [i.to(device, non_blocking=True) for i in images]
    if targets is None:
        return images
    targets = [{"boxes": t["boxes"].as_subclass(torch.Tensor).to(device), "labels": t["labels"].to(device)} for t in targets]
    return images, targets

def make_scheduler(optimizer, total_iters, warmup_iters):
    """선형 warm-up 후 cosine 감소 (최종 lr = 초기의 1%)."""
    def f(it):
        if it < warmup_iters:
            return 0.1 + 0.9 * it / max(1, warmup_iters)
        p = (it - warmup_iters) / max(1, total_iters - warmup_iters)
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * min(1.0, p)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, f)

def train_one_epoch(model, loader, optimizer, scaler, scheduler):
    """1 epoch 학습. Returns: (평균 박스 손실, 평균 분류 손실)"""
    model.train()
    sums, n = np.zeros(2), 0
    params = [p for g in optimizer.param_groups for p in g["params"]]
    for images, targets in loader:
        images, targets = _to_device(images, targets)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            ld = model(images, targets)
            loss = ld["bbox_regression"] + ld["classification"]
        if not torch.isfinite(loss):
            print("  ! 비정상 손실 → 이 배치 건너뜀"); optimizer.zero_grad(set_to_none=True); continue
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(params, 10.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        sums += [ld["bbox_regression"].item(), ld["classification"].item()]; n += 1
    return tuple(sums / max(n, 1))

@torch.no_grad()
def compute_val_loss(model, loader):
    """검증 손실: SSD 는 train 모드에서만 손실을 반환하므로 train() 후 BatchNorm 만 eval 로 고정해 계산."""
    model.train()
    for mod in model.modules():
        if isinstance(mod, nn.modules.batchnorm._BatchNorm):
            mod.eval()
    tot, n = 0.0, 0
    for images, targets in loader:
        images, targets = _to_device(images, targets)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            ld = model(images, targets)
        tot += (ld["bbox_regression"] + ld["classification"]).item(); n += 1
    model.eval()
    return tot / max(n, 1)

@torch.no_grad()
def predict_ssd(model, ids, batch_size=None, amp=None):
    """SSD 추론 → {image_id: {'boxes','scores','labels'}} (캐시 좌표계, numpy)."""
    model.eval()
    amp = USE_AMP if amp is None else amp
    out, k = {}, 0
    for images, _ in make_loader(ids, eval_tf, batch_size=batch_size):
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            dets = model(_to_device(images))
        for d in dets:
            out[ids[k]] = {key: d[key].float().cpu().numpy() if key != "labels" else d[key].cpu().numpy()
                           for key in ("boxes", "scores", "labels")}
            k += 1
    return out

def fit(model, groups, train_loader, val_ids, epochs, name, patience=None):
    """학습 + 매 epoch 검증(val loss, mAP) + 조기 종료. 최고 val mAP 가중치로 복원해 반환."""
    patience = patience or CFG["patience"]
    optimizer = torch.optim.AdamW(groups, weight_decay=CFG["weight_decay"])
    total = epochs * len(train_loader)
    scheduler = make_scheduler(optimizer, total, CFG["warmup_epochs"] * len(train_loader))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    val_loader = make_loader(val_ids, eval_tf)
    hist = {k: [] for k in ["train_box", "train_cls", "train_loss", "val_loss", "val_mAP", "val_mAP50", "lr"]}
    best, best_state, wait = -1.0, None, 0
    for ep in range(1, epochs + 1):
        t0 = time.time()
        lb, lc = train_one_epoch(model, train_loader, optimizer, scaler, scheduler)
        vl = compute_val_loss(model, val_loader)
        vm = evaluate_detections(predict_ssd(model, val_ids), val_ids)
        for k, v in zip(hist, [lb, lc, lb + lc, vl, vm["mAP"], vm["mAP50"], optimizer.param_groups[1]["lr"]]):
            hist[k].append(v)
        flag = ""
        if vm["mAP"] > best + 1e-4:
            best, wait, flag = vm["mAP"], 0, " *best"
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
        print(f"[{name}] ep{ep:02d} | train box {lb:.3f} cls {lc:.3f} | val loss {vl:.3f} | "
              f"val mAP {vm['mAP']:.4f} mAP50 {vm['mAP50']:.4f} | {time.time()-t0:.0f}s{flag}")
        if wait >= patience:
            print(f"[{name}] EarlyStopping at epoch {ep} (best val mAP {best:.4f})"); break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, hist

### 7-4. 실험 실행 · 기록 · 복원 함수
`run_experiment()` 가 하나의 실험을 끝까지 수행하고 결과를 `RESULTS` 에 저장한다. 가중치(`outputs_m7/<이름>.pt`)와 구성(`experiments.json`)을 함께 저장하므로 **커널을 재시작해도 재학습 없이 모델을 복원**할 수 있다(`load_model`).

클래스 불균형 실험용 `make_balanced_sampler()` 도 여기서 정의한다 : 이미지의 종(cat/dog) 빈도의 역수를 샘플링 가중치로 쓰는 **WeightedRandomSampler** 로, 한 epoch 안에서 고양이 이미지가 개와 비슷한 횟수로 뽑히게 한다(데이터를 버리지 않는 오버샘플링).

In [ ]:
RESULTS = globals().get("RESULTS", {})
META_PATH = OUT_DIR / "experiments.json"

def save_meta():
    meta = json.loads(META_PATH.read_text(encoding="utf-8")) if META_PATH.exists() else {}
    meta.update({n: {k: r[k] for k in ("arch", "loss", "aug", "mosaic_p", "sampler", "weights")} for n, r in RESULTS.items()})
    META_PATH.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

def make_balanced_sampler(ids, power=1.0):
    """종 빈도의 역수^power 를 가중치로 하는 WeightedRandomSampler. power=1 이면 cat:dog ≈ 1:1 로 뽑힘."""
    sp = img_df.loc[ids, "species"].values
    freq = Counter(sp)
    w = np.array([(1.0 / freq[s]) ** power for s in sp])
    return WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), num_samples=len(ids), replacement=True)

def diagnose(r):
    """train/val 지표와 학습곡선으로 과적합·과소적합을 판정."""
    tr, va, h = r["train_metrics"], r["val_metrics"], r["history"]
    gap = tr["mAP"] - va["mAP"]
    vl = np.array(h.get("val_loss") or [np.nan])
    rise = False
    if np.isfinite(vl).sum() >= 3:
        b = int(np.nanargmin(vl)); rise = (len(vl) - 1 - b) >= 2 and vl[-1] > np.nanmin(vl) * 1.15
    if tr["mAP50"] < 0.80:
        verdict = "과소적합(Underfitting)"
    elif gap > 0.10 or rise:
        verdict = "과적합(Overfitting) 경향"
    else:
        verdict = "적정 적합(Good fit)"
    detail = (f"train mAP {tr['mAP']:.4f} / val mAP {va['mAP']:.4f} (gap {gap:+.4f}), "
              f"train mAP50 {tr['mAP50']:.4f} / val mAP50 {va['mAP50']:.4f}{', val loss 재상승' if rise else ''}")
    return verdict, detail

def _diag_train_ids():
    rng = np.random.default_rng(0)
    return sorted(rng.choice(train_ids, size=min(600, len(train_ids)), replace=False).tolist())

def run_experiment(name, arch="ssd300", aug="basic", mosaic_p=0.0, sampler=None, loss="ce", epochs=None):
    """SSD 계열 실험 1개를 학습·평가·저장한다.

    Args:
        aug      : AUGS 의 키 ('basic' | 'ssd')
        mosaic_p : Mosaic 확률
        sampler  : None | 'balanced' (클래스 불균형 오버샘플링)
        loss     : 'ce' | 'focal_giou'
    """
    seed_everything()
    epochs = epochs or CFG["epochs"]["ssdlite" if arch == "ssdlite" else "ssd512" if arch == "ssd512" else "ssd"]
    model, groups = build_detector(arch, loss=loss)
    smp = make_balanced_sampler(train_ids) if sampler == "balanced" else None
    train_loader = make_loader(train_ids, AUGS[aug], shuffle=True, mosaic_p=mosaic_p, sampler=smp)
    t0 = time.time()
    model, hist = fit(model, groups, train_loader, val_ids, epochs, name)
    train_time = time.time() - t0
    val_preds = predict_ssd(model, val_ids)
    tr_ids = _diag_train_ids()
    r = dict(name=name, arch=arch, loss=loss, aug=aug, mosaic_p=mosaic_p, sampler=sampler, history=hist,
             weights=str(OUT_DIR / f"{name}.pt"),
             train_metrics=evaluate_detections(predict_ssd(model, tr_ids), tr_ids),
             val_metrics=full_metrics(val_preds, val_ids), val_preds=val_preds,
             params=count_params(model)[1], time_min=train_time / 60)
    r["verdict"], r["verdict_detail"] = diagnose(r)
    RESULTS[name] = r
    torch.save(model.state_dict(), r["weights"]); save_meta()
    vm = r["val_metrics"]
    print(f"\n▶ {name} | 학습 {r['time_min']:.1f}분 | val mAP {vm['mAP']:.4f} mAP50 {vm['mAP50']:.4f} "
          f"AP_cat {vm['AP_cat']:.4f} AP_dog {vm['AP_dog']:.4f} AP_S {vm['AP_S']:.4f}\n  진단: {r['verdict']} — {r['verdict_detail']}")
    del model; gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return r

def load_model(name):
    """저장된 가중치로 SSD 모델 복원 (재학습 없음)."""
    meta = json.loads(META_PATH.read_text(encoding="utf-8"))[name] if name not in RESULTS else RESULTS[name]
    m, _ = build_detector(meta["arch"], pretrained=False, loss=meta.get("loss", "ce"))
    m.load_state_dict(torch.load(meta["weights"], map_location=device))
    return m.eval()

def results_table(names, split="val"):
    """실험별 지표 비교표."""
    missing = [n for n in names if n not in RESULTS]
    if missing:
        raise RuntimeError(f"RESULTS 에 없는 실험: {missing} — 해당 실험 셀을 먼저 실행하세요.")
    rows = []
    for n in names:
        r = RESULTS[n]; m = r[f"{split}_metrics"]
        rows.append({"model": n, **{k: m.get(k, np.nan) for k in METRIC_COLS},
                     "train_mAP": r["train_metrics"]["mAP"], "gap(train-val mAP)": r["train_metrics"]["mAP"] - r["val_metrics"]["mAP"],
                     "params(M)": r["params"] / 1e6, "epochs": len(r["history"].get("val_mAP", [])),
                     "time(min)": r["time_min"], "진단": r["verdict"]})
    return pd.DataFrame(rows).set_index("model")

def plot_histories(names, title=""):
    """학습곡선: (1) train/val loss, (2) val mAP@[.5:.95], (3) val mAP50."""
    fig, axes = plt.subplots(1, 3, figsize=(19, 4.5))
    for i, n in enumerate(names):
        h = RESULTS[n]["history"]; c = f"C{i}"
        if not h.get("val_mAP"):
            continue
        ep = np.arange(1, len(h["val_mAP"]) + 1)
        if h.get("train_loss"):
            axes[0].plot(ep, h["train_loss"], "--", color=c)
        if h.get("val_loss") and np.isfinite(h["val_loss"]).any():
            axes[0].plot(ep, h["val_loss"], "-", color=c, label=n)
        axes[1].plot(ep, h["val_mAP"], "-o", ms=3, color=c, label=n)
        axes[2].plot(ep, h["val_mAP50"], "-o", ms=3, color=c, label=n)
    for ax, t in zip(axes, ["Loss (dashed=train, solid=val)", "Val mAP@[.5:.95]", "Val mAP@0.5"]):
        ax.set_title(t); ax.set_xlabel("epoch"); ax.legend(fontsize=8)
    plt.suptitle(title); plt.tight_layout(); plt.show()

---
## 8. 실험 1 — Baseline : SSD300-VGG16 (COCO 사전학습, 기본 증강)

이후 모든 개선 실험의 **비교 기준선**이다. 증강은 좌우 반전만 사용한다.

In [ ]:
run_experiment("ssd300_base", arch="ssd300", aug="basic")
plot_histories(["ssd300_base"], "Baseline SSD300 learning curves")

**해석 포인트**
- COCO 에서 이미 cat/dog 를 학습한 백본·박스 헤드 덕분에 **첫 1~2 epoch 만에 mAP50 이 빠르게 올라가는 것**이 일반적이다(Transfer Learning 효과).
- 증강이 약하므로 train loss 는 계속 내려가지만 val mAP 는 일찍 정체하거나 val loss 가 다시 오르는 **과적합 패턴**이 보이기 쉽다 — 위 `진단` 과 곡선으로 확인한다.
- mAP50 (대략 위치) 대비 mAP@[.5:.95] (정밀 위치)가 얼마나 낮은지가 **박스 정밀도의 개선 여지**를 보여준다.



---
## 9. 실험 2 — 고도화된 데이터 증강

**통제 조건** : 모델(SSD300)·손실·epoch·seed 동일. 증강만 바꾼다.

| 실험명 | 증강 | 확인하려는 것 |
|---|---|---|
| `ssd300_base` | basic (반전) | 기준 |
| `ssd300_aug` | SSD 표준 증강 (Photometric + ZoomOut + IoUCrop + 반전) | 크기·위치·색 다양성의 규제 효과 |
| `ssd300_mosaic` | SSD 증강 + **Mosaic(p=0.5)** | 다중·소형 객체 합성 효과 |

In [ ]:
run_experiment("ssd300_aug", arch="ssd300", aug="ssd")
run_experiment("ssd300_mosaic", arch="ssd300", aug="ssd", mosaic_p=0.5)

AUG_EXPS = ["ssd300_base", "ssd300_aug", "ssd300_mosaic"]
aug_tbl = results_table(AUG_EXPS)
display(aug_tbl)
plot_histories(AUG_EXPS, "Augmentation comparison (SSD300)")

BEST_AUG_EXP = aug_tbl["mAP"].idxmax()
BEST_AUG = dict(aug=RESULTS[BEST_AUG_EXP]["aug"], mosaic_p=RESULTS[BEST_AUG_EXP]["mosaic_p"])
print(f"\n선택된 증강: {BEST_AUG_EXP} → {BEST_AUG}  (val mAP {aug_tbl['mAP'].max():.4f})")
print((aug_tbl[["mAP", "mAP50", "mAP75", "AP_S"]] - aug_tbl.loc["ssd300_base", ["mAP", "mAP50", "mAP75", "AP_S"]])
      .add_suffix(" 개선폭").to_string())

**분석 방법**
- `gap(train-val mAP)` 과 학습곡선 : 증강이 강해지면 train 지표는 덜 오르지만(학습이 어려워짐) val 지표가 유지·향상되면 **일반화가 좋아진 것**이다.
- **ZoomOut/Mosaic 은 객체를 작게, IoUCrop 은 크게** 만든다. 따라서 `AP_S`(소형)와 `AP_L`(대형) 변화를 함께 보면 어떤 증강이 어느 크기 구간에 기여했는지 알 수 있다.
- 강한 증강은 수렴이 느려 epoch 이 짧으면 과소적합처럼 보일 수 있다. 이 경우 epoch 을 늘리면 차이가 더 벌어진다.
- 검증셋 박스가 500~600개 수준이므로 **mAP 0.005 이하 차이는 난수 수준**일 수 있음을 감안한다.

이후 실험(10~12장)은 여기서 선택된 `BEST_AUG` 를 공통으로 사용한다.



---
## 10. 실험 3 — 소형 객체 탐지 개선

### 10-1. 앵커 커버리지 분석 — "작은 얼굴은 왜 놓치는가?"
SSD 는 정답과 **IoU ≥ 0.5 인 앵커가 하나도 없으면 그 정답을 학습조차 못 한다**(양성 앵커 없음). 따라서 각 정답 박스에 대해 "가장 잘 맞는 앵커와의 IoU" 를 계산해, 크기 구간별로 **앵커 recall(최대 IoU ≥ 0.5 비율)** 을 비교한다.

- 비교 : SSD300 기본 앵커 vs SSD512 + 작은 scale 앵커(0.04 부터 시작)
- 앵커는 모델 입력 해상도(300 / 512) 좌표계에서 생성되므로, 정답 박스도 해당 해상도로 변환해서 비교한다.

In [ ]:
@torch.no_grad()
def anchor_coverage(arch, ids):
    """정답 박스별 '최적 앵커 IoU' 를 계산 (모델 입력 해상도 좌표계)."""
    m, _ = build_detector(arch, pretrained=False)
    m.eval()
    S = 512 if arch == "ssd512" else 320 if arch == "ssdlite" else 300
    x = torch.zeros(1, 3, S, S, device=device)
    feats = list(m.backbone(x).values())
    from torchvision.models.detection.image_list import ImageList
    anchors = m.anchor_generator(ImageList(x, [(S, S)]), feats)[0].cpu()
    rows = []
    for i in ids:
        g = GT[i]; H, W = g["size"]
        b = torch.as_tensor(g["boxes"]) * torch.tensor([S / W, S / H, S / W, S / H])
        best = box_iou(b, anchors).max(1).values.numpy()
        for ra, bi in zip(rel_area(g["boxes"], g["size"]), best):
            rows.append({"arch": arch, "rel_area": ra, "best_iou": bi})
    del m
    return pd.DataFrame(rows), len(anchors)

cov = []
for a in ["ssd300", "ssd512"]:
    d, na = anchor_coverage(a, train_ids)
    print(f"{a}: 앵커 수 {na:,}")
    cov.append(d)
cov = pd.concat(cov)
cov["size"] = pd.cut(cov.rel_area, [r[0] for r in AREA_RANGES.values()] + [1.01], labels=list(AREA_RANGES), right=False)
cov_tbl = cov.groupby(["size", "arch"]).best_iou.agg(mean_best_iou="mean", anchor_recall_50=lambda v: (v >= 0.5).mean()).unstack()
display(cov_tbl)

fig, ax = plt.subplots(figsize=(8, 4.5))
for a, g in cov.groupby("arch"):
    ax.scatter(g.rel_area, g.best_iou, s=6, alpha=.4, label=a)
ax.axhline(0.5, color="k", ls="--", lw=.8); ax.axvline(AREA_THR[0], color="r", ls=":", lw=1, label="small threshold")
ax.set_xscale("log"); ax.set_xlabel("GT relative area"); ax.set_ylabel("best anchor IoU"); ax.legend()
ax.set_title("Anchor coverage: best IoU between each GT box and default boxes"); plt.show()

**결과 해석** : 큰 얼굴은 두 설정 모두 최적 앵커 IoU 가 높지만, **Small 구간에서는 SSD300 기본 앵커의 IoU 가 떨어지고 0.5 미만인 박스가 생긴다**(가장 작은 앵커 scale 이 0.07, 즉 300px 입력에서 약 21px 이고 첫 특징맵 간격이 8px). SSD512 + 작은 scale(0.04 → 512px 에서 약 20px, 특징맵 64×64) 은 같은 박스에 더 촘촘하고 작은 앵커를 제공해 **앵커 recall 을 끌어올린다**.



### 10-2. SSD512 + 작은 앵커 학습
- 입력 해상도 300 → **512** : 작은 얼굴의 픽셀 수가 약 2.9배가 되어 특징이 덜 뭉개진다.
- 앵커 scale 을 SSD512 논문 값으로 축소 (위 분석 근거)
- 증강은 `BEST_AUG` (ZoomOut/Mosaic 이 포함됐다면 작은 얼굴 학습 예제도 함께 늘어남)
- 비교 대상 : 같은 증강의 SSD300 (`BEST_AUG_EXP`) — 특히 **AP_S** 변화를 본다.

In [ ]:
run_experiment("ssd512_small", arch="ssd512", **BEST_AUG)
small_tbl = results_table([BEST_AUG_EXP, "ssd512_small"])
display(small_tbl[["mAP", "mAP50", "mAP75", "AP_S", "AP_M", "AP_L", "recall", "params(M)", "time(min)", "진단"]])

fig, ax = plt.subplots(figsize=(7, 4))
small_tbl[["AP_S", "AP_M", "AP_L"]].T.plot.bar(ax=ax, rot=0)
ax.set_title("Val mAP@[.5:.95] by object size"); ax.set_ylim(0, 1); plt.show()

**해석** : 개선이 의도대로 동작했다면 `AP_S` 가 가장 크게 오르고, `AP_M`/`AP_L` 은 유지되거나 소폭 오른다. 대가는 **학습·추론 시간 증가**(입력 면적 약 2.9배)로, 16장에서 속도로 정량화한다. 만약 AP_S 개선이 미미하다면, 이 데이터셋의 "작은 얼굴"이 COCO 기준으로는 여전히 중형 이상이라 기본 앵커로도 충분히 커버된다는 뜻이다(10-1 의 앵커 recall 차이 크기와 함께 판단).


---
## 11. 실험 4 — 클래스 불균형 해소 (cat : dog ≈ 1 : 2)

3장에서 고양이 이미지가 개의 절반 정도임을 확인했다. 탐지 모델에서 불균형은 주로 **종 분류 쪽**에 나타난다 : 얼굴은 찾았지만 고양이를 개로 분류하거나(→ cat 의 FN + dog 의 FP), 고양이 점수가 낮게 보정된다.

- 조치 : **종 빈도 역수 가중 샘플링**(WeightedRandomSampler) — 한 epoch 에 cat·dog 이미지가 비슷하게 뽑히도록 오버샘플링
- 비교 : 같은 증강의 SSD300 (`BEST_AUG_EXP`) vs `ssd300_balanced`
- 평가 : 전체 mAP 뿐 아니라 **AP_cat, AP_dog, 둘의 차이**, Top-1 종 정확도

In [ ]:
run_experiment("ssd300_balanced", arch="ssd300", sampler="balanced", **BEST_AUG)

bal_names = [BEST_AUG_EXP, "ssd300_balanced"]
bal_tbl = results_table(bal_names)[["mAP", "AP_cat", "AP_dog", "top1_acc", "precision", "recall"]]
bal_tbl["AP gap(dog-cat)"] = bal_tbl["AP_dog"] - bal_tbl["AP_cat"]
for n in bal_names:   # 클래스별 AP50
    bal_tbl.loc[n, "AP50_cat"] = RESULTS[n]["val_metrics"]["AP50_cat"]
    bal_tbl.loc[n, "AP50_dog"] = RESULTS[n]["val_metrics"]["AP50_dog"]
display(bal_tbl)

# 클래스별 PR 곡선 (val, IoU 0.5)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, c in zip(axes, CLASS_IDS):
    for n in bal_names:
        rec, prec = RESULTS[n]["val_metrics"]["pr"][c]
        ax.plot(rec, prec, label=f"{n} (AP50 {RESULTS[n]['val_metrics']['AP50_' + CLASSES[c]]:.3f})")
    ax.set_title(f"PR curve @IoU0.5 — {CLASSES[c]}"); ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_xlim(0, 1.01); ax.set_ylim(0, 1.01); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**해석**
- 오버샘플링이 효과적이면 **AP_cat 이 오르고 `AP gap(dog-cat)` 이 줄어든다.** PR 곡선에서는 cat 곡선이 오른쪽 위로 이동한다(같은 정밀도에서 더 많은 고양이 얼굴을 찾음).
- AP_dog 이 약간 떨어질 수 있다 : 개 이미지의 노출 빈도가 상대적으로 줄기 때문. **전체 mAP 와 두 클래스 균형 중 무엇을 우선할지**는 서비스 목적에 따라 정한다.
- 이 데이터의 불균형(1:2)은 심하지 않아 개선폭이 작을 수 있다. 고양이 얼굴은 생김새가 품종 간에 비슷해(개보다 분산이 작음) **적은 데이터로도 잘 학습되는 편**이기 때문이다. 불균형이 클수록(예: 1:10) 이 기법의 효과가 커진다.
- 12장의 **Focal Loss** 도 손실 측면에서 불균형을 다루는 방법이다(어려운·소수 예제 가중).

---
## 12. 실험 5 — 모델 아키텍처 및 손실함수 개선

### 12-1. 손실함수 개선 : Focal Loss + GIoU (SSD300, `BEST_AUG`)
7-2 에서 설명한 손실로 교체한다. 기대 효과 : (1) Hard Negative Mining 휴리스틱 제거, 어려운 예제 집중 → 분류 안정, (2) GIoU 로 IoU 직접 최적화 → **mAP75 / mAP@[.5:.95]** 향상.

In [ ]:
run_experiment("ssd300_focal_giou", arch="ssd300", loss="focal_giou", **BEST_AUG)
display(results_table([BEST_AUG_EXP, "ssd300_focal_giou"])[["mAP", "mAP50", "mAP75", "AP_cat", "AP_dog", "mean_iou", "precision", "recall", "진단"]])
plot_histories([BEST_AUG_EXP, "ssd300_focal_giou"], "Loss function: CE+HNM+SmoothL1 vs Focal+GIoU")

**해석** : 손실 스케일이 달라 두 모델의 loss 절대값은 직접 비교하지 않고 **val mAP 곡선**으로 비교한다. GIoU 가 의도대로 동작하면 `mAP50` 보다 `mAP75`·`mean_iou`(매칭된 박스의 평균 IoU) 개선이 더 크게 나타난다. Focal Loss 는 점수 분포를 낮추는 경향이 있어, 같은 점수 임계값(0.5)에서 Recall 이 달라질 수 있다 → AP 는 임계값과 무관하므로 공정 비교는 AP 로 한다.



### 12-2. 경량 아키텍처 : SSDLite320-MobileNetV3-Large
- 백본을 VGG16 → **MobileNetV3**(depthwise separable conv, 약 8배 적은 파라미터)로, 헤드를 depthwise conv 기반 **SSDLite 헤드**로 바꾼 모바일용 SSD.
- 용량이 작아 수렴이 느리므로 epoch 을 더 길게 준다(`CFG["epochs"]["ssdlite"]`).
- 목적 : **정확도를 얼마나 잃고 속도를 얼마나 얻는가** (16장 속도 측정과 연결).

In [ ]:
run_experiment("ssdlite320", arch="ssdlite", **BEST_AUG)
display(results_table([BEST_AUG_EXP, "ssdlite320"])[["mAP", "mAP50", "mAP75", "AP_S", "params(M)", "epochs", "time(min)", "진단"]])
plot_histories([BEST_AUG_EXP, "ssdlite320"], "SSD300-VGG16 vs SSDLite320-MobileNetV3")

**해석** : SSDLite 는 파라미터가 1/8 수준이지만, 사전학습된 COCO 탐지 가중치 덕분에 이 정도 난이도(큰 객체 위주, 2클래스)의 문제에서는 mAP50 차이가 크지 않은 경우가 많다. 차이는 주로 **정밀 위치(mAP75)와 소형 객체(AP_S)** 에서 벌어진다 — 얕은 특징맵 해상도(320 입력, 첫 특징맵 20×20)가 작은 얼굴에 불리하기 때문이다.



### 12-3. (검토) YOLO 와의 비교 — 해볼 만한가?

**결론 : 비교할 가치가 있다.** 이유는 다음과 같다.

1. **같은 One-stage 계열의 현대적 기준점** : SSD(2016) 이후 YOLO 계열은 앵커-프리 헤드, 개선된 박스 손실(DFL+CIoU), Mosaic·MixUp 등 강한 증강, 자동 학습 스케줄을 갖췄다. "SSD 를 개선한 결과가 최신 탐지기 대비 어느 위치인가"를 보여주는 **상한선(reference)** 역할을 한다.
2. **경량 모델과의 공정한 비교** : YOLO nano(약 260만 파라미터)는 SSDLite(약 300만)와 크기가 비슷하다.
3. **후처리 관점의 대조군** : **YOLO11n** 은 NMS 를 쓰고, **YOLO26n** 은 학습 시 one-to-one 매칭으로 **NMS 없이(end-to-end)** 출력한다 → 15장 후처리 논의와 연결된다.

**공정성 유의사항** : YOLO 는 자체 학습 파이프라인(증강·스케줄·손실)을 쓰므로 "모델 구조만" 바꾼 통제 실험은 아니다. 그래서 **데이터 분할(train/val/test)과 평가 함수는 SSD 와 완전히 동일**하게 맞추고(같은 캐시 이미지, 같은 `evaluate_detections`), 입력 크기는 320 으로 SSD300 과 비슷하게 둔다. Ultralytics 는 **AGPL-3.0** 라이선스이므로 상업적 사용 시 라이선스를 확인해야 한다.

#### YOLO 데이터 변환
캐시된 이미지를 그대로 jpg 로 저장하고, 박스를 YOLO 형식(`class cx cy w h`, 0~1 정규화, class 0=cat / 1=dog)으로 변환한다. 같은 캐시 이미지를 쓰므로 예측 좌표가 SSD 와 같은 좌표계가 된다.

In [ ]:
YOLO_DIR = OUT_DIR / "yolo_data"

def export_yolo_dataset(root=YOLO_DIR):
    """캐시 이미지 + 박스 → Ultralytics YOLO 형식(images/{split}, labels/{split}, data.yaml)."""
    for split, ids in [("train", train_ids), ("val", val_ids), ("test", test_ids)]:
        (root / "images" / split).mkdir(parents=True, exist_ok=True)
        (root / "labels" / split).mkdir(parents=True, exist_ok=True)
        for iid in ids:
            c = CACHE[iid]; h, w = c["img"].shape[:2]
            cv2.imwrite(str(root / "images" / split / f"{iid}.jpg"), cv2.cvtColor(c["img"], cv2.COLOR_RGB2BGR),
                        [cv2.IMWRITE_JPEG_QUALITY, 95])
            lines = [f"{int(l) - 1} {(b[0] + b[2]) / 2 / w:.6f} {(b[1] + b[3]) / 2 / h:.6f} "
                     f"{(b[2] - b[0]) / w:.6f} {(b[3] - b[1]) / h:.6f}" for b, l in zip(c["boxes"], c["labels"])]
            (root / "labels" / split / f"{iid}.txt").write_text("\n".join(lines))
    yaml = (f"path: {root.resolve().as_posix()}\ntrain: images/train\nval: images/val\ntest: images/test\n"
            f"names:\n  0: cat\n  1: dog\n")
    (root / "data.yaml").write_text(yaml)
    return root / "data.yaml"

def _yolo_device():
    return 0 if device.type == "cuda" else ("mps" if device.type == "mps" else "cpu")

def _yolo_fp16(half):
    """Ultralytics 버전별 FP16 추론 인자 (신버전: quantize=16, 구버전: half=True)."""
    if not half:
        return {}
    from ultralytics.cfg import DEFAULT_CFG_DICT
    return {"quantize": 16} if "quantize" in DEFAULT_CFG_DICT else {"half": True}

def predict_yolo(model, ids, batch=32, half=None):
    """YOLO 추론 → SSD 와 같은 형식 {image_id: {'boxes','scores','labels'}} (labels: 1=cat, 2=dog)."""
    half = (device.type == "cuda") if half is None else half
    out = {}
    for k in range(0, len(ids), batch):
        chunk = ids[k:k + batch]
        imgs = [np.ascontiguousarray(CACHE[i]["img"][..., ::-1]) for i in chunk]      # Ultralytics 는 BGR ndarray 입력
        res = model.predict(imgs, imgsz=CFG["yolo_imgsz"], conf=0.01, iou=0.45, max_det=100,
                            verbose=False, device=_yolo_device(), **_yolo_fp16(half))
        for iid, r in zip(chunk, res):
            b = r.boxes
            out[iid] = {"boxes": b.xyxy.cpu().numpy(), "scores": b.conf.cpu().numpy(),
                        "labels": b.cls.cpu().numpy().astype(int) + 1}
    return out

def run_yolo_experiment(weights):
    """YOLO 학습(Ultralytics 기본 파이프라인) → 동일 평가 함수로 평가 → RESULTS 에 등록."""
    from ultralytics import YOLO
    name = Path(weights).stem.replace(".", "_")
    seed_everything()
    model = YOLO(weights if PRETRAINED else weights.replace(".pt", ".yaml"))
    t0 = time.time()
    model.train(data=str(DATA_YAML), epochs=CFG["epochs"]["yolo"], imgsz=CFG["yolo_imgsz"], batch=CFG["batch_size"],
                seed=CFG["seed"], workers=CFG["num_workers"], device=_yolo_device(),
                project=str((OUT_DIR / "yolo").resolve()), name=name, exist_ok=True,
                patience=max(10, CFG["patience"] * 2), plots=False, verbose=False, amp=USE_AMP,
                fliplr=0.5, flipud=0.0)
    train_time = time.time() - t0
    save_dir = Path(model.trainer.save_dir)
    best = save_dir / "weights" / "best.pt"
    model = YOLO(str(best))
    hist = {}
    csv = save_dir / "results.csv"
    if csv.exists():
        h = pd.read_csv(csv); h.columns = [c.strip() for c in h.columns]
        tl = [c for c in h.columns if c.startswith("train/") and c.endswith("loss")]
        vl = [c for c in h.columns if c.startswith("val/") and c.endswith("loss")]
        hist = {"train_loss": h[tl].sum(1).tolist(), "val_loss": h[vl].sum(1).tolist(),
                "val_mAP": h["metrics/mAP50-95(B)"].tolist(), "val_mAP50": h["metrics/mAP50(B)"].tolist()}
    val_preds = predict_yolo(model, val_ids)
    tr_ids = _diag_train_ids()
    r = dict(name=name, arch="yolo", loss="yolo(default)", aug="yolo(default: mosaic 등)", mosaic_p=1.0, sampler=None,
             weights=str(best), history=hist,
             train_metrics=evaluate_detections(predict_yolo(model, tr_ids), tr_ids),
             val_metrics=full_metrics(val_preds, val_ids), val_preds=val_preds,
             params=sum(p.numel() for p in model.model.parameters()), time_min=train_time / 60)
    r["verdict"], r["verdict_detail"] = diagnose(r)
    RESULTS[name] = r; save_meta()
    vm = r["val_metrics"]
    print(f"▶ {name} | 학습 {r['time_min']:.1f}분 | val mAP {vm['mAP']:.4f} mAP50 {vm['mAP50']:.4f} | {r['verdict']}")
    return r

def load_yolo(name):
    from ultralytics import YOLO
    w = RESULTS[name]["weights"] if name in RESULTS else json.loads(META_PATH.read_text(encoding="utf-8"))[name]["weights"]
    return YOLO(w)

YOLO_EXPS = []
if RUN_YOLO:
    try:
        import ultralytics
        print("ultralytics", ultralytics.__version__)
        DATA_YAML = export_yolo_dataset()
        for w in CFG["yolo_models"]:
            print("=" * 100)
            try:
                YOLO_EXPS.append(run_yolo_experiment(w)["name"])
            except Exception as e:
                print(f"  ! {w} 실패 → 건너뜀: {type(e).__name__}: {e}")
    except ImportError:
        print("ultralytics 미설치 → YOLO 비교 생략 (pip install ultralytics)")
if YOLO_EXPS:
    display(results_table([BEST_AUG_EXP, "ssdlite320"] + YOLO_EXPS)[
        ["mAP", "mAP50", "mAP75", "AP_S", "AP_cat", "AP_dog", "params(M)", "time(min)", "진단"]])

**해석** : YOLO 는 SSD 와 비슷하거나 더 작은 크기로 **mAP75 / mAP@[.5:.95]** 에서 앞서는 경우가 많다(앵커-프리 + DFL/IoU 계열 손실로 박스가 더 정밀함). 반면 사전학습 COCO SSD 도 이 데이터처럼 객체가 크고 클래스가 2개인 문제에서는 mAP50 기준으로 큰 차이 없이 경쟁력이 있다. YOLO26 은 NMS 없이도 중복 박스가 적게 나오는지(15장 후처리 비교)도 함께 본다.

> YOLO 의 학습곡선(`val_mAP`)은 Ultralytics 자체 검증값이고, 표의 val/test 지표는 SSD 와 같은 `evaluate_detections` 로 다시 계산한 값이다.

---
## 13. 검증셋 기준 전체 모델 비교 · 과적합/과소적합 진단

In [ ]:
SSD_EXPS = [n for n in ["ssd300_base", "ssd300_aug", "ssd300_mosaic", "ssd512_small", "ssd300_balanced",
                        "ssd300_focal_giou", "ssdlite320"] if n in RESULTS]
ALL_EXPS = SSD_EXPS + [n for n in YOLO_EXPS if n in RESULTS]
val_tbl = results_table(ALL_EXPS)
display(val_tbl.style.format(precision=4).highlight_max(subset=["mAP", "mAP50", "mAP75", "AP_S", "f1"], color="#d3f9d8"))

fig, axes = plt.subplots(1, 2, figsize=(19, 4.8))
val_tbl[["mAP50", "mAP75", "mAP", "AP_S"]].plot.bar(ax=axes[0], rot=30); axes[0].set_ylim(0, 1)
axes[0].set_title("Validation AP by model")
val_tbl[["precision", "recall", "f1", "top1_acc"]].plot.bar(ax=axes[1], rot=30); axes[1].set_ylim(0, 1.05)
axes[1].set_title(f"Validation operating point (score ≥ {CFG['op_score']}, IoU 0.5)")
plt.tight_layout(); plt.show()
plot_histories(ALL_EXPS, "All models — validation curves")

print("[과적합/과소적합 진단]")
for n in ALL_EXPS:
    print(f" - {n:20s}: {RESULTS[n]['verdict']}\n   {RESULTS[n]['verdict_detail']}")

FINAL = val_tbl["mAP"].idxmax()                                    # 전체 최종 모델 (val mAP@[.5:.95])
FINAL_SSD = val_tbl.loc[SSD_EXPS, "mAP"].idxmax()                  # 후처리 실험 대상 SSD
print(f"\n→ 최종 모델(val mAP 최고): {FINAL} | SSD 계열 최고: {FINAL_SSD}")

### 결과 해석 가이드 (지표 → 판단 근거)

1. **증강 효과 (base → aug → mosaic)** : `gap(train-val mAP)` 가 줄고 val mAP 가 오르면 증강이 과적합을 억제한 것이다. Baseline 은 train mAP 가 val 보다 크게 높아(격차 > 0.1) **과적합 경향**으로 진단되기 쉽다.
2. **소형 객체 (ssd512_small)** : `AP_S` 열을 본다. 전체 mAP 는 Small 비율(25%)만큼만 반영되므로, AP_S 가 크게 올라도 mAP 상승폭은 작을 수 있다.
3. **불균형 (ssd300_balanced)** : `AP_cat` 과 `AP_dog` 의 차이가 줄었는지 본다.
4. **손실함수 (ssd300_focal_giou)** : `mAP75`, `mean_iou` 가 오르면 박스 정밀도가 개선된 것이다.
5. **경량 모델 (ssdlite320, YOLO nano)** : 파라미터 대비 성능(16장 속도와 함께)으로 평가한다. 용량이 작은 모델은 train mAP 자체가 낮게 머물러 **과소적합**으로 진단될 수 있다 — 이 경우 epoch 증가·입력 해상도 증가가 처방이다.
6. **선택 기준** : 최종 모델은 **val mAP@[.5:.95]** 로 고른다. 이 값은 검출 여부(Recall)·오탐(Precision)·위치 정밀도를 모두 반영하며, 점수 임계값과 무관해 모델 자체의 능력을 비교하기에 적합하다.

---
## 14. 최종 테스트 — 테스트셋 평가 (모델별 Precision / Recall / AP / mAP)

지금까지 한 번도 사용하지 않은 **test 셋**으로 모든 모델을 평가한다. 후처리 설정은 모든 SSD 가 동일(score 0.01, NMS 0.45, 최대 100개), Precision/Recall 은 `score ≥ 0.5`, IoU 0.5 기준이다.

In [ ]:
def predict_any(name, ids):
    """실험 이름으로 모델을 복원해 예측 (SSD / YOLO 공통 인터페이스)."""
    if RESULTS[name]["arch"] == "yolo":
        return predict_yolo(load_yolo(name), ids)
    m = load_model(name)
    p = predict_ssd(m, ids)
    del m
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return p

TEST_PREDS, TEST_METRICS = {}, {}
for n in ALL_EXPS:
    TEST_PREDS[n] = predict_any(n, test_ids)
    TEST_METRICS[n] = full_metrics(TEST_PREDS[n], test_ids)

test_tbl = pd.DataFrame({n: {k: TEST_METRICS[n][k] for k in METRIC_COLS + ["TP", "FP", "FN"]} for n in ALL_EXPS}).T
test_tbl["val_mAP"] = val_tbl.loc[ALL_EXPS, "mAP"]
test_tbl["mAP gap(val-test)"] = test_tbl["val_mAP"] - test_tbl["mAP"]
test_tbl = test_tbl.astype(float)
test_tbl.to_csv(OUT_DIR / "test_results.csv")

def _hl(row):
    return ["background-color: #fff3bf" if row.name == FINAL else "" for _ in row]
display(test_tbl.style.apply(_hl, axis=1).format(precision=4)
        .highlight_max(subset=["mAP", "mAP50", "mAP75", "AP_S", "f1"], color="#d3f9d8"))

# 클래스별 AP 상세표 (IoU 0.5 / 0.75 / 0.5:0.95)
cls_rows = []
for n in ALL_EXPS:
    for c in CLASS_IDS:
        ap = TEST_METRICS[n]["AP"][c]
        cls_rows.append({"model": n, "class": CLASSES[c], "AP50": ap[0], "AP75": ap[list(IOU_THRS).index(0.75)],
                         "AP@[.5:.95]": np.nanmean(ap)})
cls_tbl = pd.DataFrame(cls_rows).pivot(index="model", columns="class").loc[ALL_EXPS]
print("\n[테스트셋 클래스별 AP]")
display(cls_tbl.style.format(precision=4))

In [ ]:
show = list(dict.fromkeys(["ssd300_base", BEST_AUG_EXP, FINAL_SSD, FINAL] + [n for n in ALL_EXPS if n in YOLO_EXPS or n == "ssdlite320"]))
fig, axes = plt.subplots(1, 3, figsize=(21, 5))
# (1) IoU 임계값별 mAP — 위치 정밀도 비교
for n in show:
    APs = np.nanmean(np.array([TEST_METRICS[n]["AP"][c] for c in CLASS_IDS]), axis=0)
    axes[0].plot(IOU_THRS, APs, "-o", ms=3, label=n)
axes[0].set_title("Test mAP vs IoU threshold"); axes[0].set_xlabel("IoU threshold"); axes[0].set_ylabel("mAP"); axes[0].legend(fontsize=8)
# (2)(3) 클래스별 PR 곡선 @IoU0.5
for ax, c in zip(axes[1:], CLASS_IDS):
    for n in show:
        rec, prec = TEST_METRICS[n]["pr"][c]
        ax.plot(rec, prec, label=f"{n} (AP50 {TEST_METRICS[n]['AP'][c][0]:.3f})")
    ax.set_title(f"Test PR curve @IoU0.5 — {CLASSES[c]}"); ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_xlim(0, 1.01); ax.set_ylim(0, 1.01); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(19, 4.5))
test_tbl.loc[ALL_EXPS, ["AP_S", "AP_M", "AP_L"]].plot.bar(ax=axes[0], rot=30); axes[0].set_title("Test mAP@[.5:.95] by object size"); axes[0].set_ylim(0, 1)
test_tbl.loc[ALL_EXPS, ["TP", "FP", "FN"]].plot.bar(ax=axes[1], rot=30); axes[1].set_title(f"Test TP / FP / FN (score ≥ {CFG['op_score']}, IoU 0.5)")
plt.tight_layout(); plt.show()

f = test_tbl.loc[FINAL]
print(f"■ 최종 모델 {FINAL} (test): mAP@[.5:.95] {f.mAP:.4f} | mAP50 {f.mAP50:.4f} | mAP75 {f.mAP75:.4f} | "
      f"Precision {f.precision:.4f} | Recall {f.recall:.4f} | F1 {f.f1:.4f} | mean IoU {f.mean_iou:.4f}")

### 테스트 결과 해석 가이드

1. **val vs test 격차 (`mAP gap(val-test)`)** : 이 노트북의 test 는 val 과 같은 분포(같은 데이터에서 층화 분할)이므로 격차가 작아야 정상이다. 격차가 크면 val 에 과하게 맞춘 선택(조기 종료 시점 등)의 영향이거나 표본 수(수백 장)에 따른 변동이다. (미션 6 의 X-ray 처럼 test 가 다른 분포였던 경우와 대비된다.)
2. **mAP vs IoU 임계값 곡선** : 곡선이 0.5 에서는 비슷하고 **0.75 이후 빠르게 떨어지는 모델은 "찾기는 하지만 박스가 덜 정밀한" 모델**이다. GIoU 손실·고해상도 모델은 오른쪽 구간의 하락이 완만해야 한다.
3. **PR 곡선** : 오른쪽 위에 붙을수록 좋다. 곡선 끝(최대 Recall)이 1 에 못 미치면 **아예 검출되지 않은 얼굴(FN)** 이 있다는 뜻이다(점수 임계값을 낮춰도 못 찾음).
4. **TP/FP/FN** : 운영점에서 FP 가 많으면 → 15장의 후처리(Class-agnostic NMS, max-det)로 줄일 수 있는지, FN 이 많으면 → 점수 임계값을 낮추거나 소형 객체 개선이 필요한지 판단한다.
5. **클래스별 AP** : cat/dog 중 낮은 쪽이 개선 우선순위다.

---
## 15. 후처리(Post-processing) 개선 — NMS · Soft-NMS · Class-agnostic · max-det · 점수 임계값

모델을 다시 학습하지 않고 **출력 정리 방식만 바꿔** 성능·속도를 개선한다. 대상은 SSD 계열 최고 모델(`FINAL_SSD`).

1. NMS 를 끈 상태(IoU 1.0)로 **후처리 전 후보 박스(raw candidates)** 를 한 번만 뽑아 두고
2. 여러 후처리 방식을 같은 후보에 적용해 **val** 에서 비교한 뒤 (CPU 에서 후처리 시간도 측정)
3. val 에서 고른 설정을 **test** 에 그대로 적용한다.

| 방식 | 설명 | 기대 효과 |
|---|---|---|
| NMS (클래스별) 0.3 / 0.45 / 0.6 | 같은 클래스 안에서 IoU 가 임계값 이상인 박스 제거 | 임계값 ↓ → 중복 ↓ (가까운 두 얼굴은 하나로 합쳐질 위험) |
| **Class-agnostic NMS** | cat/dog 구분 없이 NMS | 한 얼굴에 cat 박스와 dog 박스가 **동시에 남는 오탐 제거** (이 문제에서 흔함) |
| **Soft-NMS** (Gaussian) | 겹치는 박스를 지우지 않고 점수를 IoU 에 따라 감쇠 | 가까운 두 마리를 덜 놓침, AP(특히 Recall 측) 개선 가능 |
| **max_det = 1 / 3** | 이미지당 최대 박스 수 제한 | 대부분 얼굴 1개라는 **데이터 사전지식** 활용 → Precision ↑ |
| 점수 임계값 최적화 | val 에서 F1 최대 임계값 선택 | 운영점 Precision–Recall 균형 |

In [ ]:
@torch.no_grad()
def raw_candidates(model, ids):
    """NMS 를 끄고(nms_thresh=1.0) 점수 0.01 이상 후보를 클래스별 상위 400개까지 뽑는다."""
    saved = (model.nms_thresh, model.detections_per_img, model.topk_candidates)
    model.nms_thresh, model.detections_per_img, model.topk_candidates = 1.0, 800, 400
    out = predict_ssd(model, ids)
    model.nms_thresh, model.detections_per_img, model.topk_candidates = saved
    return out

def soft_nms_np(boxes, scores, sigma=0.5, score_thr=0.01, max_cand=200):
    """Gaussian Soft-NMS: 선택된 박스와 겹친 정도만큼 나머지 점수를 exp(-IoU²/σ) 로 감쇠.
    Returns: (원래 인덱스 배열, 감쇠된 점수 배열)"""
    order = np.argsort(-scores)[:max_cand]
    b, s, idx = boxes[order].copy(), scores[order].copy(), order.copy()
    keep_i, keep_s = [], []
    while len(s):
        j = int(np.argmax(s))
        keep_i.append(idx[j]); keep_s.append(s[j])
        s = s * np.exp(-(iou_np(b[j:j + 1], b)[0] ** 2) / sigma)
        mask = np.ones(len(s), bool); mask[j] = False; mask &= s >= score_thr
        b, s, idx = b[mask], s[mask], idx[mask]
    return np.array(keep_i, int), np.array(keep_s)

def postprocess(raw, method="nms", iou=0.45, agnostic=False, max_det=100, score_thr=0.01):
    """raw 후보에 후처리를 적용. Returns: (예측 dict, 이미지당 후처리 시간 ms, 이미지당 평균 박스 수)"""
    out, t0 = {}, time.perf_counter()
    for iid, p in raw.items():
        b, s, l = p["boxes"], p["scores"], p["labels"]
        m = s >= score_thr; b, s, l = b[m], s[m], l[m]
        if method == "nms":
            bt, st, lt = torch.as_tensor(b), torch.as_tensor(s), torch.as_tensor(l)
            keep = (nms(bt, st, iou) if agnostic else batched_nms(bt, st, lt, iou)).numpy()
            b, s, l = b[keep], s[keep], l[keep]
        elif method == "soft":
            if agnostic:
                k, s = soft_nms_np(b, s); b, l = b[k], l[k]
            else:
                parts = [(np.where(l == c)[0], soft_nms_np(b[l == c], s[l == c])) for c in CLASS_IDS]
                b = np.concatenate([b[ix][k] for ix, (k, _) in parts]) if parts else b
                l = np.concatenate([l[ix][k] for ix, (k, _) in parts])
                s = np.concatenate([ns for _, (_, ns) in parts])
        o = np.argsort(-s)[:max_det]
        out[iid] = {"boxes": b[o], "scores": s[o], "labels": l[o]}
    ms = (time.perf_counter() - t0) / max(len(raw), 1) * 1000
    return out, ms, np.mean([len(v["scores"]) for v in out.values()])

PP_VARIANTS = {
    "NMS 0.45 (default)":           dict(method="nms", iou=0.45),
    "NMS 0.30":                  dict(method="nms", iou=0.30),
    "NMS 0.60":                  dict(method="nms", iou=0.60),
    "Class-agnostic NMS 0.45":   dict(method="nms", iou=0.45, agnostic=True),
    "Soft-NMS (agnostic)":       dict(method="soft", agnostic=True),
    "Agnostic NMS + max_det=3":  dict(method="nms", iou=0.45, agnostic=True, max_det=3),
    "Agnostic NMS + max_det=1":  dict(method="nms", iou=0.45, agnostic=True, max_det=1),
}
pp_model = load_model(FINAL_SSD)
RAW = {"val": raw_candidates(pp_model, val_ids), "test": raw_candidates(pp_model, test_ids)}
print(f"후처리 대상: {FINAL_SSD} | NMS 전 후보 수(이미지당 평균): {np.mean([len(v['scores']) for v in RAW['val'].values()]):.0f}")

pp_rows, PP_PREDS = {}, {}
for k, kw in PP_VARIANTS.items():
    pv, ms, nd = postprocess(RAW["val"], **kw)
    m = evaluate_detections(pv, val_ids)
    pp_rows[k] = {"mAP": m["mAP"], "mAP50": m["mAP50"], "precision": m["precision"], "recall": m["recall"],
                  "f1": m["f1"], "FP": m["FP"], "FN": m["FN"], "dets/img": nd, "postproc ms/img (CPU)": ms}
    PP_PREDS[k] = pv
pp_val = pd.DataFrame(pp_rows).T
display(pp_val.style.format(precision=4).highlight_max(subset=["mAP", "f1"], color="#d3f9d8")
        .highlight_min(subset=["FP", "postproc ms/img (CPU)"], color="#d3f9d8"))
PP_BEST_MAP = pp_val["mAP"].astype(float).idxmax()
PP_BEST_F1 = pp_val["f1"].astype(float).idxmax()
print(f"val 기준 → mAP 최고: {PP_BEST_MAP} | 운영점 F1 최고: {PP_BEST_F1}")

**결과 해석**
- **Class-agnostic NMS** 는 한 얼굴 위에 cat·dog 박스가 겹쳐 남는 경우를 정리해 **FP 를 크게 줄이는 것**이 보통이다. 종 분류가 헷갈리는 얼굴(점수가 두 클래스에 나뉨)에서 높은 쪽 하나만 남기 때문이다.
- **max_det=1** 은 Precision 을 최대로 올리지만, 두 마리가 있는 이미지에서는 Recall 을 잃는다. 이 데이터처럼 "이미지당 1얼굴"이 대부분이면 운영점 F1 이 가장 좋을 수 있다 — **데이터 사전지식이 후처리 설계를 바꾸는 사례**.
- **Soft-NMS** 는 박스를 지우지 않으므로 dets/img 가 많고 후처리 시간이 길다(파이썬 구현 기준). AP 에는 유리할 수 있지만 운영점 Precision 은 낮아질 수 있다.
- **mAP 는 낮은 점수 박스까지 포함한 순위 지표**라 후처리 변화에 둔감한 편이고, **운영점 지표(P/R/F1, FP)** 가 더 크게 움직인다 → 후처리는 서비스 목적(놓치지 않기 vs 오탐 줄이기)에 맞춰 고른다.

### 15-2. 점수 임계값 최적화 (val → test)

In [ ]:
pv_best, _, _ = postprocess(RAW["val"], **PP_VARIANTS[PP_BEST_F1])
thr_grid = np.round(np.arange(0.1, 0.91, 0.05), 2)
sweep = pd.DataFrame({t: {k: evaluate_detections(pv_best, val_ids, op_score=t)[k] for k in ["precision", "recall", "f1"]}
                      for t in thr_grid}).T
BEST_THR = float(sweep["f1"].idxmax())
ax = sweep.plot(figsize=(8, 4), marker="o", ms=3); ax.axvline(BEST_THR, color="k", ls="--", lw=.8)
ax.set_title(f"Val P/R/F1 vs score threshold ({PP_BEST_F1}) → best {BEST_THR:.2f}"); ax.set_xlabel("score threshold"); plt.show()

# test 적용 : 기본 후처리 vs val 에서 고른 후처리(+임계값)
pp_test_rows = {}
for label, kw, thr in [("기본 (NMS 0.45, thr 0.5)", PP_VARIANTS["NMS 0.45 (default)"], CFG["op_score"]),
                       (f"{PP_BEST_MAP} (thr 0.5)", PP_VARIANTS[PP_BEST_MAP], CFG["op_score"]),
                       (f"{PP_BEST_F1} (thr {BEST_THR:.2f})", PP_VARIANTS[PP_BEST_F1], BEST_THR)]:
    pt, ms, nd = postprocess(RAW["test"], **kw)
    m = evaluate_detections(pt, test_ids, op_score=thr)
    pp_test_rows[label] = {"mAP": m["mAP"], "mAP50": m["mAP50"], "precision": m["precision"], "recall": m["recall"],
                           "f1": m["f1"], "FP": m["FP"], "FN": m["FN"], "dets/img": nd, "postproc ms/img": ms}
pp_test = pd.DataFrame(pp_test_rows).T
display(pp_test.style.format(precision=4))

**해석** : val 에서 정한 후처리·임계값을 test 에 그대로 적용했을 때도 FP 감소와 F1 향상이 유지되면, 후처리 개선이 **재학습 없이 얻는 실질적 성능 향상**임을 확인한 것이다. 임계값은 val 에서만 정했으므로 test 결과는 낙관적으로 부풀려지지 않았다.



---
## 16. 추론 속도 · 경량화 · 가속화 실험

| 실험 | 내용 | 확인할 것 |
|---|---|---|
| 16-1 모델 크기 · 연산량 | 파라미터, 가중치 파일 크기, GFLOPs | 경량 구조의 이론적 이득 |
| 16-2 End-to-end 지연시간 | 배치 1, 전처리+네트워크+후처리 포함 실측 (FP32 / FP16) | 실제 서비스 응답 시간 |
| 16-3 네트워크 처리량 | 배치 32, FP32 / FP16 / channels_last / torch.compile | GPU 가속 기법별 효과 |
| 16-4 입력 해상도 축소 | SSDLite 를 256 / 320 / 384 입력으로 평가 (재학습 없음) | 속도–정확도 트레이드오프 |
| 16-5 ONNX Runtime + INT8 양자화 | CPU 배포 시 PyTorch vs ORT FP32 vs ORT INT8 (속도·mAP·크기) | 배포 최적화 효과 |
| 16-6 속도–정확도 종합 | test mAP vs 지연시간 산점도 | 파레토 최적 모델 |

> 측정값은 GPU 종류·드라이버·동시 부하에 따라 달라진다. **같은 세션에서 같은 방식**으로 잰 상대 비교로 해석한다. GPU 시간은 `torch.cuda.synchronize()` 로 비동기 실행을 기다린 뒤 측정한다.

In [ ]:
def _sync():
    if device.type == "cuda":
        torch.cuda.synchronize()

def time_fn(fn, n_warmup=5, n_iter=None):
    """fn() 평균 실행 시간(ms). warm-up 으로 cuDNN autotune·캐시 효과를 제외."""
    n_iter = n_iter or (5 if QUICK_MODE else 30)
    for _ in range(n_warmup):
        fn()
    _sync(); t0 = time.perf_counter()
    for _ in range(n_iter):
        fn()
    _sync()
    return (time.perf_counter() - t0) / n_iter * 1000

class SSDRaw(nn.Module):
    """SSD 의 순수 네트워크 부분(백본+헤드)만 분리한 모듈. 입력은 이미 정규화·리사이즈된 (B,3,S,S).
    처리량 측정, FLOPs 계산, ONNX 내보내기에 사용한다 (전처리·디코딩·NMS 는 제외)."""
    def __init__(self, ssd):
        super().__init__()
        self.backbone, self.head = ssd.backbone, ssd.head
    def forward(self, x):
        feats = list(self.backbone(x).values())
        out = self.head(feats)
        return out["bbox_regression"], out["cls_logits"]

def input_size(m):
    return m.transform.fixed_size[0] if getattr(m.transform, "fixed_size", None) else m.transform.min_size[0]

def gflops(module, S):
    """FlopCounterMode 로 1장 forward 의 GFLOPs (곱·합 각각 1 FLOP) 측정."""
    try:
        from torch.utils.flop_counter import FlopCounterMode
        x = torch.zeros(1, 3, S, S, device=device)
        with FlopCounterMode(display=False) as fc, torch.no_grad():
            module(x)
        return fc.get_total_flops() / 1e9
    except Exception as e:
        return np.nan

SPEED_IMGS = [torch.from_numpy(CACHE[i]["img"]).permute(2, 0, 1).float().div(255) for i in test_ids[:20]]
SPEED = {}
SPEED_SSD = list(dict.fromkeys([n for n in ["ssd300_base", FINAL_SSD, "ssd512_small", "ssdlite320"] if n in RESULTS]))
for n in SPEED_SSD:
    m = load_model(n); raw = SSDRaw(m).eval(); S = input_size(m)
    row = {"params(M)": RESULTS[n]["params"] / 1e6, "file(MB)": Path(RESULTS[n]["weights"]).stat().st_size / 1e6,
           "input": S, "GFLOPs": gflops(raw, S)}
    k = [0]
    def e2e(amp):
        def f():
            with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                m([SPEED_IMGS[k[0] % len(SPEED_IMGS)].to(device)])
            k[0] += 1
        return f
    row["e2e ms (bs1, FP32)"] = time_fn(e2e(False))
    if device.type == "cuda":
        row["e2e ms (bs1, FP16)"] = time_fn(e2e(True))
    # 네트워크만, 배치 32 처리량
    B = 8 if device.type == "cpu" else 32
    x = torch.randn(B, 3, S, S, device=device)
    def net(model_, inp, amp):
        def f():
            with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                model_(inp)
        return f
    row[f"net img/s (bs{B}, FP32)"] = B / time_fn(net(raw, x, False)) * 1000
    if device.type == "cuda":
        row[f"net img/s (bs{B}, FP16)"] = B / time_fn(net(raw, x, True)) * 1000
        raw_cl = copy.deepcopy(raw).to(memory_format=torch.channels_last)
        row[f"net img/s (bs{B}, FP16+channels_last)"] = B / time_fn(net(raw_cl, x.to(memory_format=torch.channels_last), True)) * 1000
        try:
            raw_c = torch.compile(copy.deepcopy(raw))
            row[f"net img/s (bs{B}, FP16+compile)"] = B / time_fn(net(raw_c, x, True), n_warmup=3) * 1000
        except Exception as e:
            print(f"  torch.compile 생략 ({n}): {type(e).__name__}")
    SPEED[n] = row
    del m, raw; gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

# YOLO end-to-end (Ultralytics predict, 전처리+NMS 포함)
for n in [y for y in YOLO_EXPS if y in RESULTS]:
    ym = load_yolo(n)
    imgs = [np.ascontiguousarray(CACHE[i]["img"][..., ::-1]) for i in test_ids[:20]]
    k = [0]
    def yf(half):
        def f():
            ym.predict(imgs[k[0] % len(imgs)], imgsz=CFG["yolo_imgsz"], conf=0.25, verbose=False,
                       device=_yolo_device(), **_yolo_fp16(half)); k[0] += 1
        return f
    row = {"params(M)": RESULTS[n]["params"] / 1e6, "file(MB)": Path(RESULTS[n]["weights"]).stat().st_size / 1e6,
           "input": CFG["yolo_imgsz"], "e2e ms (bs1, FP32)": time_fn(yf(False))}
    if device.type == "cuda":
        row["e2e ms (bs1, FP16)"] = time_fn(yf(True))
    try:
        row["GFLOPs"] = gflops(ym.model.to(device).eval(), CFG["yolo_imgsz"])
    except Exception:
        pass
    SPEED[n] = row
speed_tbl = pd.DataFrame(SPEED).T
display(speed_tbl.style.format(precision=2))

**결과 해석**
- **모델 크기 · GFLOPs** : SSDLite·YOLO nano 는 SSD300-VGG16 대비 파라미터·연산량이 한 자릿수 배 작다. SSD512 는 파라미터는 같지만 GFLOPs 가 약 3배다(입력 면적 비례).
- **End-to-end 지연시간(bs1)** : 배치 1 에서는 GPU 가 충분히 바쁘지 않아 전처리·후처리·파이썬 오버헤드 비중이 커진다. 그래서 GFLOPs 차이만큼 시간이 줄지는 않는다.
- **FP16(AMP)** : Tensor Core 를 쓰는 GPU 에서 처리량이 크게 오른다(대개 1.5~3배). 출력 차이는 mAP 에 거의 영향이 없다.
- **channels_last / torch.compile** : 합성곱 메모리 배치·커널 융합으로 추가 이득이 있을 수 있다(환경에 따라 0~30%). Windows 등 compile 이 지원되지 않는 환경에서는 자동으로 생략된다.

### 16-4. 입력 해상도 축소 — SSDLite 속도–정확도 트레이드오프 (재학습 없음)
SSD 의 앵커는 특징맵 크기에 맞춰 자동 생성되므로, 학습한 모델을 다른 입력 해상도로 바로 실행할 수 있다. 해상도를 줄이면 속도는 오르고, 특히 **작은 얼굴의 AP** 가 떨어질 것이다.

In [ ]:
res_rows = {}
res_target = "ssdlite320" if "ssdlite320" in RESULTS else FINAL_SSD
for S in [256, 320, 384]:
    m = load_model(res_target)
    m.transform = GeneralizedRCNNTransform(min_size=S, max_size=S, image_mean=m.transform.image_mean,
                                           image_std=m.transform.image_std, size_divisible=1, fixed_size=(S, S))
    pv = predict_ssd(m, val_ids); mt = full_metrics(pv, val_ids)
    x = torch.randn(8, 3, S, S, device=device); raw = SSDRaw(m).eval()
    def f():
        with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            raw(x)
    res_rows[S] = {"val mAP": mt["mAP"], "val mAP50": mt["mAP50"], "val AP_S": mt["AP_S"],
                   "net ms/img (bs8)": time_fn(f) / 8, "GFLOPs": gflops(raw, S)}
    del m, raw
res_tbl = pd.DataFrame(res_rows).T.rename_axis("input size")
display(res_tbl.style.format(precision=4))
print(f"대상 모델: {res_target} (학습 해상도 {320 if res_target == 'ssdlite320' else 'native'})")

**해석** : 학습 해상도(320)에서 가장 잘 맞는 것이 보통이며, 256 으로 줄이면 연산량이 약 36% 줄지만 AP_S 가 먼저 떨어진다. 384 는 작은 얼굴에 약간 유리할 수 있으나 학습 때 보지 못한 스케일이라 이득이 제한적이다 → **배포 해상도를 바꾸려면 그 해상도로 재학습(또는 multi-scale 학습)** 하는 것이 정석이다.

### 16-5. ONNX Runtime 변환 + INT8 정적 양자화 (CPU 배포 시나리오)
GPU 가 없는 서버·엣지 환경을 가정해, 네트워크 부분(백본+헤드)을 **ONNX** 로 내보내고 **ONNX Runtime** 으로 실행한다. 전처리(정규화·리사이즈)와 후처리(앵커 디코딩·NMS)는 torchvision 코드를 그대로 재사용해 **결과가 PyTorch 와 같은 형식**이 되게 하고, 그 결과로 mAP 까지 측정한다.

- **ORT FP32** : 그래프 최적화(연산 융합 등)로 PyTorch CPU 대비 속도 이득 기대, 정확도 동일
- **ORT INT8 (정적 양자화, QDQ)** : train 이미지 일부로 활성값 범위를 보정(calibration)한 뒤 가중치·활성값을 8bit 정수로 → **파일 크기 약 1/4, CPU 속도 향상**, 정확도 소폭 손실 가능

In [ ]:
class OnnxSSDPipeline:
    """torch 전처리(model.transform) → ONNX Runtime(백본+헤드) → torch 후처리(디코딩·NMS).

    SSD 모델과 같은 호출 방식 pipeline(list_of_images) → list_of_detections 을 제공한다."""
    def __init__(self, torch_model, onnx_path, providers=("CPUExecutionProvider",)):
        import onnxruntime as ort
        self.m = torch_model.cpu().eval()
        so = ort.SessionOptions(); so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        self.sess = ort.InferenceSession(str(onnx_path), so, providers=list(providers))
        self._anchor = None

    @torch.no_grad()
    def __call__(self, images):
        from torchvision.models.detection.image_list import ImageList
        orig = [tuple(im.shape[-2:]) for im in images]
        il, _ = self.m.transform([im.cpu() for im in images])
        br, cl = self.sess.run(None, {"images": il.tensors.numpy().astype(np.float32)})
        if self._anchor is None:   # 앵커는 입력 크기가 고정이므로 한 번만 계산
            feats = list(self.m.backbone(il.tensors[:1]).values())
            self._anchor = self.m.anchor_generator(ImageList(il.tensors[:1], il.image_sizes[:1]), feats)[0]
        heads = {"bbox_regression": torch.from_numpy(br), "cls_logits": torch.from_numpy(cl)}
        dets = self.m.postprocess_detections(heads, [self._anchor] * len(images), il.image_sizes)
        return self.m.transform.postprocess(dets, il.image_sizes, orig)

def export_onnx(model, path):
    """SSDRaw(백본+헤드)를 ONNX 로 내보낸다 (배치 크기 동적)."""
    raw = SSDRaw(copy.deepcopy(model).cpu().eval()).eval()
    S = input_size(model)
    x = torch.randn(1, 3, S, S)
    kw = dict(input_names=["images"], output_names=["bbox_regression", "cls_logits"], opset_version=17,
              dynamic_axes={"images": {0: "batch"}, "bbox_regression": {0: "batch"}, "cls_logits": {0: "batch"}})
    try:
        torch.onnx.export(raw, (x,), str(path), dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(raw, (x,), str(path), **kw)
    return path

def quantize_onnx_int8(fp32_path, int8_path, model, calib_ids, n=64):
    """train 이미지 n 장으로 보정한 INT8 정적 양자화 (QDQ, per-channel 가중치)."""
    from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType
    class Reader(CalibrationDataReader):
        def __init__(self):
            self.it = iter(calib_ids[:n])
        def get_next(self):
            iid = next(self.it, None)
            if iid is None:
                return None
            im = torch.from_numpy(CACHE[iid]["img"]).permute(2, 0, 1).float().div(255)
            il, _ = model.cpu().transform([im])
            return {"images": il.tensors.numpy().astype(np.float32)}
    quantize_static(str(fp32_path), str(int8_path), Reader(), quant_format=QuantFormat.QDQ,
                    activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8, per_channel=True)
    return int8_path

@torch.no_grad()
def predict_with(fn, ids, bs=8):
    """임의의 탐지 함수 fn(list_of_images) 로 예측 dict 생성 (CPU)."""
    out = {}
    for k in range(0, len(ids), bs):
        chunk = ids[k:k + bs]
        ims = [torch.from_numpy(CACHE[i]["img"]).permute(2, 0, 1).float().div(255) for i in chunk]
        for iid, d in zip(chunk, fn(ims)):
            out[iid] = {"boxes": d["boxes"].numpy(), "scores": d["scores"].numpy(), "labels": d["labels"].numpy()}
    return out

ONNX_TBL = None
if RUN_ONNX:
    try:
        import onnx, onnxruntime as ort
        print("onnxruntime", ort.__version__, "| providers:", ort.get_available_providers())
        eval_ids = test_ids if not QUICK_MODE else test_ids[:40]
        rows = {}
        for n in list(dict.fromkeys([n for n in ["ssdlite320", FINAL_SSD] if n in RESULTS])):
            tm = load_model(n).cpu().eval()
            fp32 = export_onnx(tm, OUT_DIR / f"{n}_fp32.onnx")
            int8 = quantize_onnx_int8(fp32, OUT_DIR / f"{n}_int8.onnx", tm, train_ids)
            one = [torch.from_numpy(CACHE[test_ids[0]]["img"]).permute(2, 0, 1).float().div(255)]
            variants = {"PyTorch FP32 (CPU)": tm, "ORT FP32 (CPU)": OnnxSSDPipeline(tm, fp32),
                        "ORT INT8 (CPU)": OnnxSSDPipeline(tm, int8)}
            # 정확도 일치 확인 (PyTorch vs ORT FP32 의 네트워크 출력)
            raw = SSDRaw(tm).eval()
            with torch.no_grad():
                il, _ = tm.transform(one)
                ref = raw(il.tensors)[1].numpy()
            got = variants["ORT FP32 (CPU)"].sess.run(None, {"images": il.tensors.numpy()})[1]
            print(f"[{n}] PyTorch vs ORT FP32 cls_logits 최대 오차: {np.abs(ref - got).max():.2e}")
            for vn, fn in variants.items():
                ms = time_fn(lambda: fn(one), n_warmup=3, n_iter=5 if QUICK_MODE else 20)
                mt = evaluate_detections(predict_with(fn, eval_ids), eval_ids)
                size = (Path(RESULTS[n]["weights"]).stat().st_size if vn.startswith("PyTorch")
                        else (fp32 if "FP32" in vn else int8).stat().st_size) / 1e6
                rows[(n, vn)] = {"CPU e2e ms (bs1)": ms, "test mAP": mt["mAP"], "test mAP50": mt["mAP50"], "file(MB)": size}
            del tm
        ONNX_TBL = pd.DataFrame(rows).T
        _base = {n: ONNX_TBL.loc[(n, "PyTorch FP32 (CPU)"), "CPU e2e ms (bs1)"] for n in ONNX_TBL.index.get_level_values(0)}
        ONNX_TBL["speedup(vs PyTorch CPU)"] = [_base[n] / v for (n, _), v in zip(ONNX_TBL.index, ONNX_TBL["CPU e2e ms (bs1)"])]
        display(ONNX_TBL.style.format(precision=4))
    except ImportError:
        print("onnx / onnxruntime 미설치 → 생략 (pip install onnx onnxruntime)")
    except Exception as e:
        print(f"ONNX 실험 생략: {type(e).__name__}: {e}")

**결과 해석**
- **최대 오차**가 1e-4 수준이면 ONNX 변환이 수치적으로 정확하다는 뜻이고, 따라서 **ORT FP32 의 mAP 는 PyTorch 와 같다.**
- **ORT FP32** 는 연산 융합·메모리 계획 최적화로 CPU 에서 PyTorch 보다 빠른 경우가 많다(모델·CPU 에 따라 1.2~2배).
- **ORT INT8** 은 파일이 약 1/4 로 줄고, VNNI 등 정수 명령을 지원하는 CPU 에서 추가로 빨라진다. 정확도 손실(mAP 하락폭)이 0.01 안팎이면 배포용으로 수용 가능하다. 손실이 크면 보정 이미지 수를 늘리거나, 민감한 층(첫 conv, 헤드)을 FP32 로 남기는 **혼합 정밀도 양자화**, 또는 **양자화 인지 학습(QAT)** 을 고려한다.
- 지연시간에는 torch 전·후처리 시간이 포함되어 있어, 네트워크만의 가속 비율보다는 작게 나온다.

### 16-6. 속도–정확도 종합 (파레토 비교)

In [ ]:
lat_col = "e2e ms (bs1, FP16)" if "e2e ms (bs1, FP16)" in speed_tbl else "e2e ms (bs1, FP32)"
tradeoff = speed_tbl[[lat_col, "params(M)"]].join(test_tbl[["mAP", "mAP50"]], how="inner")
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(tradeoff[lat_col], tradeoff["mAP"], s=tradeoff["params(M)"] * 8 + 20, alpha=.6)
for n, r in tradeoff.iterrows():
    ax.annotate(n, (r[lat_col], r["mAP"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel(f"{lat_col}  (lower = faster)"); ax.set_ylabel("Test mAP@[.5:.95]")
ax.set_title("Speed vs accuracy (bubble size = params)"); plt.show()

# 파레토 최적: 자신보다 빠르면서 정확한 모델이 없는 모델
pareto = [n for n, r in tradeoff.iterrows()
          if not ((tradeoff[lat_col] <= r[lat_col]) & (tradeoff["mAP"] >= r["mAP"]) &
                  ((tradeoff[lat_col] < r[lat_col]) | (tradeoff["mAP"] > r["mAP"]))).any()]
display(tradeoff.sort_values(lat_col).style.format(precision=4))
print("파레토 최적 모델 :", pareto)

**해석** : 왼쪽 위(빠르고 정확)에 있는 모델이 좋다. 파레토 최적 목록에 든 모델 중에서 **서비스 요구사항(예: 30 FPS 이상, mAP 0.6 이상)** 을 만족하는 것을 고른다. 서버(GPU) 배포라면 정확도 우선으로 SSD512/YOLO 계열, 모바일·CPU 배포라면 SSDLite/YOLO nano + INT8 이 현실적인 선택이 된다.



---
## 17. 예측 시각화 · 오류 분석

### 17-1. 테스트 이미지 예측 (최종 모델)
초록 점선 = 정답, 실선 = 예측(주황 cat / 파랑 dog, 점수 ≥ 0.5, class-agnostic NMS 적용).

In [ ]:
def clean_preds(p, thr=CFG["op_score"], iou=0.45, max_det=5):
    """시각화용 정리: 점수 필터 + class-agnostic NMS + 상위 max_det."""
    out = {}
    for iid, d in p.items():
        m = d["scores"] >= thr
        b, s, l = d["boxes"][m], d["scores"][m], d["labels"][m]
        if len(s):
            k = nms(torch.as_tensor(b), torch.as_tensor(s), iou).numpy()[:max_det]; b, s, l = b[k], s[k], l[k]
        out[iid] = {"boxes": b, "scores": s, "labels": l}
    return out

FINAL_VIS = clean_preds(TEST_PREDS[FINAL])
rng = np.random.default_rng(1)
sel = rng.choice(test_ids, size=min(12, len(test_ids)), replace=False)
fig, axes = plt.subplots(2, 6, figsize=(22, 8))
for ax, iid in zip(axes.flat, sel):
    g, p = GT[iid], FINAL_VIS[iid]
    draw_boxes(ax, CACHE[iid]["img"], g["boxes"], g["labels"], color="#40c057", lw=1.5)
    for pa in ax.patches[-len(g["boxes"]):]:
        pa.set_linestyle("--")
    ious = iou_np(p["boxes"], g["boxes"]).max(1) if len(p["boxes"]) else []
    draw_boxes(ax, CACHE[iid]["img"], p["boxes"], p["labels"], p["scores"],
               title=f"{img_df.loc[iid, 'breed']} | IoU " + ", ".join(f"{v:.2f}" for v in ious))
plt.suptitle(f"Test predictions — {FINAL}"); plt.tight_layout(); plt.show()

### 17-2. 오류 유형 분석
테스트 이미지마다 최고 점수 박스(정리 후)를 정답과 비교해 오류를 분류한다.

| 유형 (표기) | 조건 | 의미 |
|---|---|---|
| 정답 (`correct`) | IoU ≥ 0.5, 종 일치 | |
| **위치 오류** (`localization`) | 0.1 ≤ IoU < 0.5 | 얼굴 일부만 / 몸 전체를 잡음 |
| **종 오분류** (`wrong species`) | IoU ≥ 0.5 지만 cat↔dog 틀림 | 위치는 맞고 분류가 틀림 |
| **미탐** (`missed(FN)`) | 점수 ≥ 0.5 박스 없음 또는 IoU < 0.1 | 얼굴을 못 찾음 |

In [ ]:
def error_type(iid, preds):
    g, p = GT[iid], preds[iid]
    if len(p["scores"]) == 0:
        return "missed(FN)", 0.0
    j = int(np.argmax(p["scores"])); iou = iou_np(p["boxes"][j:j + 1], g["boxes"]).max()
    gl = g["labels"][int(iou_np(p["boxes"][j:j + 1], g["boxes"]).argmax())]
    if iou < 0.1:
        return "missed(FN)", iou
    if iou < 0.5:
        return "localization", iou
    return ("correct" if int(p["labels"][j]) == int(gl) else "wrong species"), iou

err = pd.DataFrame([{"image_id": i, "breed": img_df.loc[i, "breed"], "species": img_df.loc[i, "species"],
                     "rel_area": rel_area(GT[i]["boxes"], GT[i]["size"]).max(),
                     **dict(zip(["type", "iou"], error_type(i, FINAL_VIS)))} for i in test_ids])
display(pd.crosstab(err.species, err.type, margins=True))
err["size"] = pd.cut(err.rel_area, [r[0] for r in AREA_RANGES.values()] + [1.01], labels=list(AREA_RANGES), right=False)
display(pd.crosstab(err["size"], err.type, normalize="index").style.format("{:.1%}"))
worst_breeds = err.assign(ok=err.type == "correct").groupby("breed").ok.mean().sort_values().head(8)
print("정답률 낮은 품종 Top8:\n", worst_breeds.round(3).to_string())

bad_ids = err[err.type != "correct"].sort_values("iou").image_id.tolist()[:12]
if bad_ids:
    fig, axes = plt.subplots(2, 6, figsize=(22, 8))
    for ax in axes.flat:
        ax.axis("off")
    for ax, iid in zip(axes.flat, bad_ids):
        g, p = GT[iid], FINAL_VIS[iid]
        draw_boxes(ax, CACHE[iid]["img"], g["boxes"], g["labels"], color="#40c057", lw=1.5)
        draw_boxes(ax, CACHE[iid]["img"], p["boxes"], p["labels"], p["scores"],
                   title=f"{err.set_index('image_id').loc[iid, 'type']} | {img_df.loc[iid, 'breed']}")
    plt.suptitle("Failure cases (green = GT)"); plt.tight_layout(); plt.show()

**해석 포인트**
- **크기별 오류율** : Small 구간에서 미탐 비율이 높다면 → 10장의 고해상도·작은 앵커 방향이 맞고, 추가로 tiling/SAHI(이미지를 잘라 확대 추론) 같은 기법을 고려할 수 있다.
- **종 오분류** : 얼굴이 털에 가려졌거나, 고양이 같은 개(예: 작은 장모종) 처럼 사람도 헷갈리는 사례가 많다. 정답률이 낮은 품종은 데이터 추가·품종별 가중치의 근거가 된다.
- **위치 오류** : 얼굴이 아닌 몸 전체를 잡거나, 옆모습(pose)에서 박스 경계가 모호한 경우가 흔하다. 정답 박스 자체의 모호성도 원인이므로, IoU 0.5 기준 지표와 0.75 기준 지표를 함께 보는 것이 중요하다.

### 17-3. 공식 `test.txt` 이미지 추론 (정답 박스 없음 — 정성 평가)
데이터셋이 제공한 test 목록 이미지는 얼굴 박스 정답이 없으므로 지표 대신 **실제 추론 결과를 눈으로 확인**한다.

In [ ]:
if demo_ids:
    demo_sel = list(np.random.default_rng(7).choice(demo_ids, size=min(12, len(demo_ids)), replace=False))
    for iid in demo_sel:
        if iid not in CACHE:
            s = load_sample(iid)
            if s is not None:
                CACHE[iid] = s
                GT[iid] = {"boxes": np.zeros((0, 4), np.float32), "labels": np.zeros(0, np.int64),
                           "difficult": np.zeros(0, bool), "size": s["img"].shape[:2]}
    demo_sel = [i for i in demo_sel if i in CACHE]
    demo_pred = clean_preds(predict_any(FINAL, demo_sel))
    fig, axes = plt.subplots(2, 6, figsize=(22, 8))
    for ax in axes.flat:
        ax.axis("off")
    for ax, iid in zip(axes.flat, demo_sel):
        p = demo_pred[iid]
        draw_boxes(ax, CACHE[iid]["img"], p["boxes"], p["labels"], p["scores"],
                   title=f"{img_df.loc[iid, 'breed']} (true species: {img_df.loc[iid, 'species']})")
    plt.suptitle(f"Official test.txt images (no GT boxes) — {FINAL}"); plt.tight_layout(); plt.show()
    sp_hit = np.mean([len(demo_pred[i]["labels"]) > 0 and CLASSES[int(demo_pred[i]["labels"][0])] == img_df.loc[i, "species"]
                      for i in demo_sel])
    print(f"(참고) 위 이미지들의 Top-1 종 일치율: {sp_hit:.1%} — 종 라벨은 list.txt 에 있으므로 확인 가능")
else:
    print("정답 없는 공식 test 이미지가 없습니다 (test 목록 전체에 XML 이 있거나 목록 파일 없음).")

**해석** : 박스 정답은 없지만 `list.txt` 의 종 라벨로 **Top-1 종 일치율**은 확인할 수 있다. 학습에 쓰지 않은 공식 test 이미지에서도 얼굴 위치와 종을 안정적으로 찾는지 정성적으로 확인한다.

---
## 18. 결론

In [ ]:
summary_names = list(dict.fromkeys(["ssd300_base", BEST_AUG_EXP, "ssd512_small", "ssd300_balanced",
                                    "ssd300_focal_giou", "ssdlite320"] + YOLO_EXPS + [FINAL]))
summary_names = [n for n in summary_names if n in test_tbl.index]
summary = test_tbl.loc[summary_names, ["mAP", "mAP50", "mAP75", "AP_cat", "AP_dog", "AP_S", "precision", "recall", "f1"]]
display(summary.style.highlight_max(axis=0, color="#d3f9d8").format(precision=4))

b, f = test_tbl.loc["ssd300_base"], test_tbl.loc[FINAL]
def d(a, b_):
    return f"{a - b_:+.4f}"
print("■  요약")
print(f" 1) 증강 선택: {BEST_AUG_EXP} {BEST_AUG} | test mAP base {b.mAP:.4f} → {test_tbl.loc[BEST_AUG_EXP, 'mAP']:.4f}")
if "ssd512_small" in test_tbl.index:
    print(f" 2) 소형 객체: AP_S {test_tbl.loc[BEST_AUG_EXP, 'AP_S']:.4f} → {test_tbl.loc['ssd512_small', 'AP_S']:.4f} (SSD512+작은 앵커)")
if "ssd300_balanced" in test_tbl.index:
    r0, r1 = test_tbl.loc[BEST_AUG_EXP], test_tbl.loc["ssd300_balanced"]
    print(f" 3) 불균형: AP_cat {r0.AP_cat:.4f} → {r1.AP_cat:.4f}, AP gap(dog-cat) {r0.AP_dog - r0.AP_cat:+.4f} → {r1.AP_dog - r1.AP_cat:+.4f}")
if "ssd300_focal_giou" in test_tbl.index:
    r1 = test_tbl.loc["ssd300_focal_giou"]
    print(f" 4) 손실(Focal+GIoU): mAP75 {test_tbl.loc[BEST_AUG_EXP, 'mAP75']:.4f} → {r1.mAP75:.4f}, mean IoU {test_tbl.loc[BEST_AUG_EXP, 'mean_iou']:.4f} → {r1.mean_iou:.4f}")
print(f" 5) 후처리({FINAL_SSD}): 기본 FP {int(pp_test.iloc[0].FP)} → {int(pp_test.iloc[-1].FP)}, "
      f"F1 {pp_test.iloc[0].f1:.4f} → {pp_test.iloc[-1].f1:.4f}  [{pp_test.index[-1]}]")
print(f" 6) 속도: 파레토 최적 {pareto}")
print(f" 7) 검증셋 진단: " + ", ".join(f"{n}={RESULTS[n]['verdict']}" for n in ALL_EXPS))
print(f" 8) 최종 모델: {FINAL}")
print(f"    test mAP@[.5:.95] {f.mAP:.4f} | mAP50 {f.mAP50:.4f} | mAP75 {f.mAP75:.4f} | Precision {f.precision:.4f} | "
      f"Recall {f.recall:.4f} | F1 {f.f1:.4f}")
print(f"    Baseline 대비: mAP {d(f.mAP, b.mAP)}, mAP75 {d(f.mAP75, b.mAP75)}, AP_S {d(f.AP_S, b.AP_S)}, F1 {d(f.f1, b.f1)}")

### 정리

**1. 데이터 & 평가 설계**
- 얼굴 박스는 공식 trainval 이미지에만 있으므로, 박스 보유 이미지를 **품종 층화 train/val/test** 로 나눠 정량 평가하고, 공식 test 목록은 정답 없는 추론 데모에 사용했다.
- 평가는 직접 구현한 COCO 방식 평가기로 **IoU, Precision/Recall(운영점), 클래스별 AP, mAP@0.5 / 0.75 / [.5:.95], 크기별 AP** 를 모든 모델(SSD·YOLO)에 동일하게 적용했다. 모든 선택(증강·모델·후처리·임계값)은 val 에서 하고 test 는 마지막에만 사용했다.

**2. SSD 와 개선 실험**
- **증강** : SSD 표준 증강(Photometric·ZoomOut·IoUCrop)과 Mosaic 은 크기·위치·문맥 다양성을 늘려 과적합을 줄인다 — train/val 격차와 학습곡선으로 확인.
- **소형 객체** : 앵커 커버리지 분석으로 작은 얼굴에서 기본 앵커의 IoU 가 부족함을 확인하고, **SSD512 + 작은 앵커**로 AP_S 를 개선(대신 연산량 약 3배).
- **클래스 불균형** : 종 역빈도 오버샘플링으로 소수 클래스(cat) AP 와 두 클래스 균형을 비교.
- **손실·아키텍처** : Focal + GIoU 로 IoU 를 직접 최적화해 높은 IoU 구간(mAP75)을 겨냥하고, SSDLite-MobileNetV3 로 정확도–경량화 트레이드오프를 확인했다. YOLO(11n/26n)는 최신 One-stage 탐지기의 기준점으로 같은 분할·평가로 비교했다.

**3. 후처리 · 추론 최적화**
- **Class-agnostic NMS, max-det** 같은 후처리만으로 재학습 없이 FP 를 줄이고 운영점 F1 을 개선할 수 있다(이미지당 얼굴 1개라는 데이터 특성 활용).
- **FP16, channels_last, torch.compile** 로 GPU 처리량을, **ONNX Runtime + INT8 양자화**로 CPU 지연시간·모델 크기를 줄이고 정확도 손실을 mAP 로 정량화했다.

**4. 한계와 다음 단계**
- 데이터가 약 3.7천 장으로 작고 test 가 같은 분포라, 다른 촬영 환경(CCTV, 실내 저조도)으로의 일반화는 별도 검증이 필요하다.
- 추가 개선 아이디어 : multi-scale 학습, K-Fold 앙상블 / WBF(Weighted Boxes Fusion), 품종(37클래스) 탐지로 확장, 얼굴 랜드마크 동시 학습, QAT(양자화 인지 학습), TensorRT 변환, 작은 얼굴 대상 SAHI(tiled inference).

In [ ]:
# 결과물 저장: 지표표 + 예측 확률(재학습 없이 표·그림을 다시 만들 수 있음)
val_tbl.to_csv(OUT_DIR / "val_results.csv")
with open(OUT_DIR / "predictions.pkl", "wb") as fp:
    pickle.dump({"val": {n: RESULTS[n]["val_preds"] for n in ALL_EXPS}, "test": TEST_PREDS,
                 "splits": {"train": train_ids, "val": val_ids, "test": test_ids},
                 "area_ranges": AREA_RANGES, "final": FINAL}, fp)
print("저장된 파일:")
for p in sorted(OUT_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.name:40s} {p.stat().st_size/1e6:8.1f} MB")

---
## 19. (참고) 재학습 없이 결과 다시 보기

커널을 재시작한 뒤에도 **1~6장 셀(설정·데이터 로드·평가 함수)만 실행**하면, 저장된 `predictions.pkl` 로 모든 모델의 테스트 지표표를 바로 다시 만들 수 있다(학습·추론 없음). 개별 모델이 필요하면 `load_model("<실험명>")`(7장 함수 정의 후)으로 가중치를 복원한다.

In [ ]:
def reload_test_table(path=OUT_DIR / "predictions.pkl"):
    """저장된 예측으로 test 지표표 재계산 (학습·추론 없음)."""
    with open(path, "rb") as fp:
        pack = pickle.load(fp)
    ids = [i for i in pack["splits"]["test"] if i in GT]
    rows = {n: {k: v for k, v in full_metrics(p, ids).items() if k in METRIC_COLS} for n, p in pack["test"].items()}
    return pd.DataFrame(rows).T, pack["final"]

if (OUT_DIR / "predictions.pkl").exists():
    _tbl, _final = reload_test_table()
    print("최종 모델:", _final)
    display(_tbl.style.format(precision=4))